In [ ]:
from pathlib import Path
import sys

# Localizar la raiz del repositorio desde Jupyter o terminal
RAIZ_REPO = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "notebooks").is_dir() and (p / "datos").is_dir()),
    None
)
if RAIZ_REPO is None:
    raise FileNotFoundError("Ejecuta desde el repositorio caso-itam")

sys.path.insert(0, str(RAIZ_REPO / "src"))


Dado que los años 2022,...,2025 son estructuralmente iguales que al 2021 (mismos formatos csv, misma codificación, mismas llaves, mismo módulo de victimización, mismos nombres y códigos para los reactivos, 16 alcaldías, etc) en este cuaderno haremos el mix de los 5 años

### Funciones para cargar y verificar un año de la ENVIPE

In [ ]:
# Todo lo que descubrimos en el cuaderno de 2021, convertido en funciones
# para aplicarlo igual a cada año. Si un año trae algo distinto, las
# verificaciones lo van a mostrar.

from pathlib import Path
import numpy as np
import pandas as pd

CODIFICACION    = "latin-1"
LLAVE_PERSONA   = ["UPM", "VIV_SEL", "HOGAR", "R_SEL"]
DELITOS_HOGAR   = ["01", "02", "03", "04"]
DELITOS_PERSONA = [f"{i:02d}" for i in range(5, 16)]
VAR_VECES = {c: f"AP6_6_{c}" for c in DELITOS_HOGAR}            # veces que ocurrió (hogar)
VAR_VECES.update({c: f"AP7_4_{c}" for c in DELITOS_PERSONA})    # veces que ocurrió (persona)


def buscar_archivo(carpeta, prefijo):
    """Encuentra el CSV que empieza con 'prefijo' (p. ej. 'TPer_Vic2').
    Así no importa si se llama TPer_Vic2.csv o TPer_Vic2_1.csv.
    Si encuentra cero o más de uno, se detiene con un error claro,
    en lugar de leer el archivo equivocado sin avisar."""
    candidatos = sorted(carpeta.rglob(f"{prefijo}*.csv"))
    if len(candidatos) != 1:
        raise FileNotFoundError(
            f"Esperaba 1 archivo '{prefijo}*.csv' en {carpeta}, "
            f"encontré {len(candidatos)}: {[c.name for c in candidatos]}")
    return candidatos[0]


def cargar_incidentes(carpeta):
    """Construye la tabla de incidentes de un año (equivale a las
    Celdas 3 a 6 de 2021). Devuelve (inc, per_vic2)."""
    carpeta  = Path(carpeta)
    mod_vic  = pd.read_csv(buscar_archivo(carpeta, "TMod_Vic"),  dtype=str,
                           encoding=CODIFICACION, index_col=0)
    per_vic2 = pd.read_csv(buscar_archivo(carpeta, "TPer_Vic2"), dtype=str,
                           encoding=CODIFICACION, index_col=0)

    # Traer de la persona: factores, residencia y las "veces" de cada delito
    columnas = (LLAVE_PERSONA + ["FAC_ELE", "FAC_HOG", "CVE_ENT", "NOM_ENT", "CVE_MUN", "NOM_MUN"]
                + list(VAR_VECES.values()))
    residencia = {"CVE_ENT": "ENT_RES", "NOM_ENT": "NOM_ENT_RES",
                  "CVE_MUN": "MUN_RES", "NOM_MUN": "NOM_MUN_RES"}
    inc = mod_vic.merge(per_vic2[columnas].rename(columns=residencia),
                        on=LLAVE_PERSONA, how="left", validate="many_to_one")

    # "veces" del tipo de delito de cada incidente (para verificar FAC_DEL)
    inc["veces"] = np.nan
    for codigo, variable in VAR_VECES.items():
        fila = inc["BPCOD"] == codigo
        inc.loc[fila, "veces"] = pd.to_numeric(inc.loc[fila, variable], errors="coerce")
    inc = inc.drop(columns=list(VAR_VECES.values()))   # ya no necesitamos las 15 columnas

    for col in ["FAC_DEL", "FAC_ELE", "FAC_HOG"]:
        inc[col] = inc[col].astype(float)

    # Definiciones verificadas en 2021 (Celda 6)
    inc["denuncio"] = (inc["BP1_20"] == "1") | (inc["BP1_21"] == "1")
    inc["carpeta"]  = inc["denuncio"] & (inc["BP1_24"] == "1")
    inc["todos"]    = True
    return inc, per_vic2


def verificar_estructura(inc, per_vic2):
    """Repite los chequeos que en 2021 hicimos uno por uno.
    Cualquier valor distinto del esperado es una señal para detenerse."""
    es_hogar     = inc["BPCOD"].isin(DELITOS_HOGAR)
    factor_base  = np.where(es_hogar, inc["FAC_HOG"], inc["FAC_ELE"])
    reconstruido = factor_base * np.maximum(1, inc["veces"] / 5)
    dif_rel      = (inc["FAC_DEL"] - reconstruido).abs() / inc["FAC_DEL"]
    claves_cdmx  = sorted(per_vic2.loc[per_vic2["CVE_ENT"] == "09", "CVE_MUN"].unique())

    chequeos = {
        "incidentes":                                    len(inc),
        "incidentes sin persona (debe ser 0)":           int(inc["FAC_ELE"].isna().sum()),
        "incidentes sin 'veces' (debe ser 0)":           int(inc["veces"].isna().sum()),
        "llave de incidente duplicada (debe ser 0)":     int(inc.duplicated(LLAVE_PERSONA + ["BPCOD", "ND_TIPO"]).sum()),
        "TD_TIPO = mín(veces, 5)":                       bool((inc["TD_TIPO"].astype(int) == inc["veces"].clip(upper=5)).all()),
        "máximo de TD_TIPO":                             int(inc["TD_TIPO"].astype(int).max()),
        "FAC_DEL: dif. relativa máx. vs fórmula":        f"{dif_rel.max():.2%}",
        "catálogo BPCOD":                                ", ".join(sorted(inc["BPCOD"].unique())),
        "BP1_21 fuera de delitos del hogar (debe ser 0)": int((~es_hogar & inc["BP1_21"].notna()).sum()),
        "valores de BP1_20":                             ", ".join(sorted(inc["BP1_20"].dropna().unique())),
        "valores de BP1_24":                             ", ".join(sorted(inc["BP1_24"].dropna().unique())),
        "alcaldías CDMX (deben ser 16, 002–017)":        f"{len(claves_cdmx)}: {claves_cdmx[0]}–{claves_cdmx[-1]}",
    }
    for nombre, valor in chequeos.items():
        print(f"  {nombre:<52} {valor}")


# --- Carpetas de cada año ---
BASE = RAIZ_REPO / "datos" / "crudos" / "ENVIPE"
# Una carpeta por año; rglob se encarga de buscar dentro, haya o no subcarpeta.
CARPETAS = {anio: BASE / f"bd_envipe_{anio}_csv" for anio in [2021, 2022, 2023, 2024, 2025]}

datos = {}   # aquí guardaremos (inc, per_vic2) de cada año
for anio, carpeta in CARPETAS.items():
    print(f"\n===== ENVIPE {anio} =====")
    datos[anio] = cargar_incidentes(carpeta)
    verificar_estructura(*datos[anio])

### Piloto de cada año contra las cifras publicadas

In [ ]:
# Funciones de las Celdas 6 a 9 de 2021, generalizadas.
# Al final arma UNA tabla con todos los años lado a lado:
# lo que calculamos (ENVIPE) junto a lo que publica el INEGI.

Z_90 = 1.645   # intervalos al 90%, como los publica el INEGI


def razon_con_error(inc, per_vic2, numerador, denominador, dominio):
    """Igual que en 2021 (Celda 7), con un cambio: recibe per_vic2
    para sacar de ahí la lista completa de UPM de ESE año."""
    w = inc["FAC_DEL"]
    y = w * inc[numerador]   * dominio
    x = w * inc[denominador] * dominio
    razon = y.sum() / x.sum()
    z = (y - razon * x) / x.sum()

    z_upm = z.groupby([inc["EST_DIS"], inc["UPM_DIS"]]).sum()
    upms  = per_vic2[["EST_DIS", "UPM_DIS"]].drop_duplicates()       # TODAS las UPM, con o sin incidentes
    z_upm = z_upm.reindex(pd.MultiIndex.from_frame(upms), fill_value=0.0)

    por_estrato    = z_upm.groupby(level="EST_DIS")
    n_h            = por_estrato.size()
    suma_cuadrados = por_estrato.apply(lambda s: ((s - s.mean()) ** 2).sum())
    varianza       = (n_h / (n_h - 1) * suma_cuadrados).sum()
    return razon, np.sqrt(varianza)


def resumen_piloto(inc, per_vic2):
    """Calcula las cifras que el INEGI publica, con NUESTRAS definiciones:
    sin vandalismo, factor FAC_DEL, CDMX por lugar de OCURRENCIA (BP1_2C)."""
    sin_vandalismo = inc["BPCOD"] != "03"
    nacional = sin_vandalismo
    cdmx     = sin_vandalismo & (inc["BP1_2C"] == "09")
    w = inc["FAC_DEL"]

    # Función "interna": solo existe dentro de resumen_piloto.
    # Ahorra repetir la misma fórmula de razón ponderada varias veces.
    def pct(num, den, dom):
        return 100 * (w * inc[num] * dom).sum() / (w * inc[den] * dom).sum()

    poblacion_cdmx = per_vic2.loc[per_vic2["CVE_ENT"] == "09", "FAC_ELE"].astype(float).sum()
    d_cdmx, ee_d = razon_con_error(inc, per_vic2, "denuncio", "todos",    cdmx)
    k_cdmx, ee_k = razon_con_error(inc, per_vic2, "carpeta",  "denuncio", cdmx)

    return {
        "Delitos nacional (millones)":     (w * nacional).sum() / 1e6,
        "d nacional (%)":                  pct("denuncio", "todos",    nacional),
        "k nacional (%)":                  pct("carpeta",  "denuncio", nacional),
        "Cifra negra nacional (%)":        100 - pct("carpeta", "todos", nacional),
        "Cifra negra CDMX (%)":            100 - pct("carpeta", "todos", cdmx),
        "Tasa delitos CDMX (por 100 mil)": 1e5 * (w * cdmx).sum() / poblacion_cdmx,
        "d CDMX (%)":                      100 * d_cdmx,
        "d CDMX: CV (%)":                  100 * ee_d / d_cdmx,
        "d CDMX: IC90 inferior (%)":       100 * (d_cdmx - Z_90 * ee_d),
        "d CDMX: IC90 superior (%)":       100 * (d_cdmx + Z_90 * ee_d),
        "k CDMX (%)":                      100 * k_cdmx,
        "k CDMX: CV (%)":                  100 * ee_k / k_cdmx,
    }


# --- Cifras publicadas por el INEGI, con su fuente ---
# 2021: presentación "Principales resultados" (diap. 15, 17, 40, 42) + tabulado 3.1
# 2022: presentación "Principales resultados" (diap. 15, 17, 40, 42); tabulado 3.1 pendiente
# 2023: presentaciones "Prevalencia e incidencia delictiva" (total de delitos, tasa por entidad)
#       y "Denuncia del delito" (diap. 7 y 9); tabulado 3.1 pendiente
# 2024: presentación "Principales resultados" (diap. 19, 22, 52, 54); tabulado 3.1 pendiente
# 2025: presentación "Principales resultados", nuevo formato (pág. 13, 23, 24).
#       No publica la cifra negra por entidad; la de la CDMX sale del tabulado 3.1
INEGI = {
    2021: {"Delitos nacional (millones)": 27.6, "d nacional (%)": 10.1, "k nacional (%)": 66.9,
           "Cifra negra nacional (%)": 93.3, "Cifra negra CDMX (%)": 92.6,
           "Tasa delitos CDMX (por 100 mil)": 53334,
           "d CDMX (%)": 11.20, "d CDMX: CV (%)": 6.78,
           "d CDMX: IC90 inferior (%)": 9.95, "d CDMX: IC90 superior (%)": 12.45,
           "k CDMX (%)": 66.07, "k CDMX: CV (%)": 4.73},
    2022: {"Delitos nacional (millones)": 28.1, "d nacional (%)": 10.1, "k nacional (%)": 67.3,
           "Cifra negra nacional (%)": 93.2, "Cifra negra CDMX (%)": 92.2,
           "Tasa delitos CDMX (por 100 mil)": 45336},
    2023: {"Delitos nacional (millones)": 26.8, "d nacional (%)": 10.9, "k nacional (%)": 69.3,
           "Cifra negra nacional (%)": 92.4, "Cifra negra CDMX (%)": 92.6,
           "Tasa delitos CDMX (por 100 mil)": 46032},
    2024: {"Delitos nacional (millones)": 31.3, "d nacional (%)": 10.4, "k nacional (%)": 68.0,
           "Cifra negra nacional (%)": 92.9, "Cifra negra CDMX (%)": 93.0,
           "Tasa delitos CDMX (por 100 mil)": 52723},
    2025: {"Delitos nacional (millones)": 33.5, "d nacional (%)": 9.6, "k nacional (%)": 70.5,
           "Cifra negra nacional (%)": 93.2,
           "Tasa delitos CDMX (por 100 mil)": 54473},
}

# --- Tabla de verificación: una columna "ENVIPE" y una "INEGI" por año ---
tabla = {}
for anio, (inc, per_vic2) in datos.items():                 # desempacamos el par en el propio for
    tabla[(anio, "ENVIPE")] = resumen_piloto(inc, per_vic2)
    tabla[(anio, "INEGI")]  = INEGI.get(anio, {})           # .get devuelve {} si el año aún no está

verificacion = pd.DataFrame(tabla)
print(verificacion.round(2).to_string(na_rep="—"))         # "—" donde el INEGI no publica la cifra

El CV de d en la CDMX sube de 6.8% en 2021 a 9.3% en 2025, y el de k de 4.7% a 6.4%. Indica que la precisión a nivel ciudad empeora en los años recientes. Conviene revisar si es porque baja la muestra de la CDMX o porque hay menos denuncias, ya que eso pesa en las estimaciones por alcaldía.

In [ ]:
# Alcaldías de OCURRENCIA: delitos ocurridos en la CDMX (BP1_2C == "09")
for anio, (inc, per_vic2) in datos.items():
    claves = inc.loc[inc["BP1_2C"] == "09", "BP1_3C"]
    validas = sorted(c for c in claves.unique() if c != "999")
    print(f"{anio}: {len(validas)} alcaldías ({validas[0]}–{validas[-1]}), "
          f"incidentes sin alcaldía (999): {(claves == '999').sum()}")

### ¿Residencia u ocurrencia?

Por **residencia**, la alcaldía es donde vive la víctima. d responde a la pregunta "de los delitos que sufren los habitantes de Iztapalapa, ¿qué fracción denuncian?". Es una pregunta sobre poblaciones.

Por **ocurrencia**, la alcaldía es donde pasó el delito. d responde a "de los delitos que ocurren en Iztapalapa, ¿qué fracción se denuncia?". Es una pregunta sobre territorios.

Las dos son legítimas, pero no responden lo mismo. 

#### ¿Qué dice la literatura?

Buil-Gil, Medina y Shlomo usaron **residencia**, y lo reconocen como limitación. Escriben que sus estimaciones son de tasas de victimización por área de residencia, no de tasas de delitos por lugar, y que eso "puede complicar los esfuerzos para comparar nuestras estimaciones con los registros policiales". Lo listan como la limitación (4), y dejan explícito que se puede resolver en investigación futura seleccionando los delitos ocurridos en el área local. Como prueba de robustez estimaron también solo con los delitos ocurridos en el área de residencia, y la correlación de Spearman con su estimación principal fue de 0.91 a nivel distrito y 0.88 a nivel vecindario.

Santillana et al. trabajan con **percepción de inseguridad**, que es naturalmente un atributo de la persona en su lugar de residencia. Ahí la pregunta ni siquiera se plantea.

#### Lo que haremos:

Ocurrencia como estimación principal y residencia como prueba de robustez, es decir, lo contrario de lo que hizo Buil-Gil. Hay tres razones:

1. Mi producto final requiere ocurrencia. El mapa corregido usa las carpetas como numerador, y las carpetas solo existen por lugar del hecho.
2. Es mi aportación. Buil-Gil señala la ocurrencia como trabajo futuro. Es lo que yo haré, y además validarlo contra cifras oficiales que se construyen igual.
3. Hace más interpretable la separación entre d y k, que es el corazón de este trabajo.

El costo de esta elección son cuatro cosas:

1. La correlación entre errores de muestreo de alcaldías distintas. El Fay-Herriot estándar supone que no existe. Vamos a medir qué fracción de la varianza viene de UPM compartidas y, si es chica, declararlo como limitación acotada.
2. Qué hacemos con los 999. Lo más simple es excluirlos y reportar su peso: alrededor del 2%, y siempre contados en el total de la ciudad.
3. Las covariables del Censo describen a los residentes, no a la población flotante. En alcaldías como Cuauhtémoc, Benito Juárez y Miguel Hidalgo, buena parte de los delitos los sufre gente que no vive ahí.
4. El dominio de ocurrencia incluye víctimas que no viven en la CDMX, sobre todo del Estado de México. Se captan con su propia muestra y su propio factor, así que la estimación es correcta, pero conviene declararlo.

In [ ]:
COLUMNAS_LLAVE = LLAVE_PERSONA + ["BPCOD", "ND_TIPO"]   # llave de incidente verificada en el día 2

COLUMNAS_A_CONSERVAR = (
    COLUMNAS_LLAVE
    + ["TD_TIPO", "veces"]                                   # tope de 5 y veces reales
    + ["BP1_20", "BP1_21", "BP1_22", "BP1_23",               # denuncia y razones
       "BP1_24", "BP1_25", "BP1_26"]                         # carpeta (crudas, para la pregunta 7)
    + ["BP1_2C", "BP1_3C"]                                   # ocurrencia (crudas)
    + ["ENT_RES", "MUN_RES", "NOM_MUN_RES"]                  # residencia (vienen de TPer_Vic2)
    + ["EST_DIS", "UPM_DIS", "FAC_DEL", "FAC_ELE", "FAC_HOG"]
    + ["denuncio", "carpeta"]                                # definiciones del día 2, ya calculadas
)

piezas = []
for anio, (inc, _) in datos.items():        # el "_" descarta per_vic2, que aquí no usamos
    # No adivinamos: si a algún año le falta una columna, nos detenemos y decimos cuál.
    faltan = [c for c in COLUMNAS_A_CONSERVAR if c not in inc.columns]
    if faltan:
        raise KeyError(f"ENVIPE {anio}: faltan las columnas {faltan}")

    pieza = inc[COLUMNAS_A_CONSERVAR].copy()
    pieza.insert(0, "anio_envipe", anio)            # edición de la encuesta
    pieza.insert(1, "anio_referencia", anio - 1)    # año en que ocurrieron los delitos
    piezas.append(pieza)

panel = pd.concat(piezas, ignore_index=True)   # una fila por incidente, cinco años

# --- Identificador único ---
# La llave UPM+VIV_SEL+... solo es única DENTRO de un año; anteponemos la edición.
panel["id_incidente"] = (panel["anio_envipe"].astype(str) + "_"
                         + panel[COLUMNAS_LLAVE].agg("_".join, axis=1))

# --- Banderas de exclusión (se marcan, no se borran) ---
panel["excl_vandalismo"] = panel["BPCOD"] == "03"
panel["cdmx_ocu"]        = panel["BP1_2C"] == "09"                       # ocurrió en la CDMX
panel["excl_alc_999"]    = panel["cdmx_ocu"] & (panel["BP1_3C"] == "999")

# --- Las dos geografías ---
# .where(condición) deja el valor donde la condición es True y pone NaN donde es False.
panel["alc_ocu"]      = panel["BP1_3C"].where(panel["cdmx_ocu"] & ~panel["excl_alc_999"])
panel["cdmx_res"]     = panel["ENT_RES"] == "09"                          # la víctima vive en la CDMX
panel["alc_res"]      = panel["MUN_RES"].where(panel["cdmx_res"])
panel["no_residente"] = panel["cdmx_ocu"] & ~panel["cdmx_res"]           # para el punto C

# --- Chequeos contra lo que ya sabemos del día 2 ---
print("Incidentes por edición:")
print(panel.groupby(["anio_envipe", "anio_referencia"]).size().to_string())
print(f"\nTotal: {len(panel):,}   (esperado: 186,329)")
print(f"id_incidente duplicados (debe ser 0): {panel['id_incidente'].duplicated().sum()}")

print("\nAlcaldías distintas por año (deben ser 16 y 16):")
print(panel.groupby("anio_envipe")
           .agg(ocurrencia=("alc_ocu", "nunique"), residencia=("alc_res", "nunique"))
           .to_string())

print("\nIncidentes CDMX con alcaldía 999, todas las categorías (esperado 71, 59, 59, 79, 90):")
print(panel.groupby("anio_envipe")["excl_alc_999"].sum().to_string())

In [ ]:
# Guardamos los dos parquets y controlamos d y k

# --- 1. Tabla de personas: el "marco" de cada año ---
# La varianza de Taylor necesita TODAS las UPM, también las que no tienen incidentes.
# Esa lista solo existe en TPer_Vic2, por eso guardamos esta segunda tabla.
COLUMNAS_PERSONA = LLAVE_PERSONA + ["EST_DIS", "UPM_DIS", "CVE_ENT", "CVE_MUN", "FAC_ELE", "FAC_HOG"]

piezas = []
for anio, (_, per_vic2) in datos.items():
    faltan = [c for c in COLUMNAS_PERSONA if c not in per_vic2.columns]
    if faltan:
        raise KeyError(f"ENVIPE {anio} (TPer_Vic2): faltan {faltan}")
    pieza = per_vic2[COLUMNAS_PERSONA].rename(columns={"CVE_ENT": "ENT_RES", "CVE_MUN": "MUN_RES"})
    pieza.insert(0, "anio_envipe", anio)
    piezas.append(pieza)

personas = pd.concat(piezas, ignore_index=True)
for col in ["FAC_ELE", "FAC_HOG"]:
    personas[col] = personas[col].astype(float)

# --- 2. Guardar ---
CARPETA_SALIDA = RAIZ_REPO / "datos" / "procesados"
CARPETA_SALIDA.mkdir(exist_ok=True)                          # la crea si no existe; si existe, no hace nada

RUTA_INCIDENTES = CARPETA_SALIDA / "envipe_incidentes.parquet"
RUTA_PERSONAS   = CARPETA_SALIDA / "envipe_personas.parquet"
panel.to_parquet(RUTA_INCIDENTES, index=False)
personas.to_parquet(RUTA_PERSONAS, index=False)
print(f"Guardado: {RUTA_INCIDENTES.name} ({len(panel):,} filas) y "
      f"{RUTA_PERSONAS.name} ({len(personas):,} filas)\n")

# --- 3. Releer desde disco ---
# Controlamos sobre lo LEÍDO, no sobre lo que está en memoria:
# si algo se pierde al guardar (tipos, ceros a la izquierda, vacíos), aquí aparece.
inc_arch = pd.read_parquet(RUTA_INCIDENTES)
per_arch = pd.read_parquet(RUTA_PERSONAS)

# --- 4. Control: misma función del día 2, dos fuentes distintas ---
# razon_con_error(inc, per_vic2, ...) solo toma de per_vic2 las columnas EST_DIS y UPM_DIS,
# así que le podemos pasar la tabla de personas del archivo sin cambiar la función.
DIA2 = {   # cifras de §7bis del diccionario, para comparar a ojo
    "d nacional": [10.07, 10.09, 10.90, 10.42, 9.58],
    "k nacional": [66.86, 67.29, 69.30, 67.99, 70.51],
    "d CDMX":     [11.20, 11.83, 11.09, 10.81, 9.97],
    "k CDMX":     [66.07, 65.92, 67.05, 64.64, 68.46],
}
PARAMETROS = {"d": ("denuncio", "todos"), "k": ("carpeta", "denuncio")}

filas = []
for i, anio in enumerate(sorted(datos)):
    # Fuente A: memoria del día 2, dominios armados con las variables crudas
    inc_mem, pv2_mem = datos[anio]
    dom_mem = {"nacional": inc_mem["BPCOD"] != "03",
               "CDMX":     (inc_mem["BPCOD"] != "03") & (inc_mem["BP1_2C"] == "09")}

    # Fuente B: el archivo, dominios armados con las BANDERAS nuevas (así también las probamos)
    inc_a = inc_arch[inc_arch["anio_envipe"] == anio].assign(todos=True)
    per_a = per_arch[per_arch["anio_envipe"] == anio]
    dom_a = {"nacional": ~inc_a["excl_vandalismo"],
             "CDMX":     ~inc_a["excl_vandalismo"] & inc_a["cdmx_ocu"]}

    for dominio in ["nacional", "CDMX"]:
        for param, (num, den) in PARAMETROS.items():
            est_m, ee_m = razon_con_error(inc_mem, pv2_mem, num, den, dom_mem[dominio])
            est_a, ee_a = razon_con_error(inc_a,   per_a,   num, den, dom_a[dominio])
            filas.append({
                "año": anio, "cifra": f"{param} {dominio}",
                "día 2 (%)":      DIA2[f"{param} {dominio}"][i],
                "archivo (%)":    100 * est_a,
                "CV archivo (%)": 100 * ee_a / est_a,
                "dif. estimación": abs(est_a - est_m),
                "dif. EE":         abs(ee_a - ee_m),
            })

control = pd.DataFrame(filas)
print(control.to_string(index=False, float_format=lambda x: f"{x:.4g}"))

# Veredicto: diferencias del orden de 1e-15 son ruido de punto flotante; cualquier cosa mayor, no.
maxima = control[["dif. estimación", "dif. EE"]].max().max()
print(f"\nDiferencia máxima memoria vs archivo: {maxima:.2e}  →  "
      f"{'IDÉNTICO' if maxima < 1e-12 else '⚠️ ALGO SE ROMPIÓ'}")

### Incidentes por alcaldía y año

In [ ]:
inc = inc_arch                         # a partir de aquí todo sale del parquet
universo = ~inc["excl_vandalismo"]     # universo del proyecto: sin vandalismo

# --- Catálogo de nombres de alcaldía ---
# TMod_Vic no trae el NOMBRE de la alcaldía de ocurrencia, solo la clave.
# Lo tomamos de la residencia (NOM_MUN_RES), que usa el mismo catálogo 002–017.
nombres = inc.loc[inc["cdmx_res"], ["MUN_RES", "NOM_MUN_RES"]].drop_duplicates()
variantes = nombres.groupby("MUN_RES")["NOM_MUN_RES"].nunique()
if (variantes > 1).any():                       # ¿algún año escribe distinto el nombre?
    print("Ojo, nombres que cambian entre años:")
    print(nombres[nombres["MUN_RES"].isin(variantes[variantes > 1].index)].to_string(index=False))
NOMBRE_ALC = (nombres.drop_duplicates("MUN_RES", keep="last")   # keep="last" = el año más reciente
                     .set_index("MUN_RES")["NOM_MUN_RES"].sort_index())

ETIQUETA_ANIO = {a: f"{a} ({a - 1})" for a in range(2021, 2026)}   # "2021 (2020)" = edición (año de referencia)
TABLAS = {}   # aquí iremos guardando las tablas que van al escrito; al final las exportamos juntas


def tabla_16x5(valores, filtro, geografia="alc_ocu", agregacion="sum"):
    """Tabla alcaldía × año.
    valores    : una Serie alineada con `inc` (lo que se suma o se cuenta)
    filtro     : Serie booleana, qué incidentes entran
    geografia  : "alc_ocu" (principal) o "alc_res" (robustez)
    agregacion : "sum" para sumar, "nunique" para contar valores distintos (p. ej. UPM)
    """
    base = pd.DataFrame({"alc": inc[geografia], "anio": inc["anio_envipe"], "v": valores})[filtro]
    t = base.pivot_table(index="alc", columns="anio", values="v", aggfunc=agregacion, fill_value=0)
    t = t.reindex(NOMBRE_ALC.index, fill_value=0)       # las 16 siempre, en orden de clave
    t.index   = [f"{c} {NOMBRE_ALC[c]}" for c in t.index]
    t.columns = [ETIQUETA_ANIO[a] for a in t.columns]
    return t


# --- Pregunta 1 ---
uno = pd.Series(1, index=inc.index)                 # sumar unos = contar filas
p1 = tabla_16x5(uno, universo & inc["alc_ocu"].notna())

# Filas de cierre: suma de alcaldías, los 999 y el total CDMX.
# El total debe coincidir con el día 2: 3,512 en 2021 y 3,029 en 2022 (diccionario §6–§7).
sin_alc = universo & inc["excl_alc_999"]
total   = universo & inc["cdmx_ocu"]
p1.loc["Suma de las 16 alcaldías"] = p1.sum()
p1.loc["Sin alcaldía (999)"] = inc[sin_alc].groupby("anio_envipe").size().reindex(range(2021, 2026), fill_value=0).values
p1.loc["Total CDMX (ocurrencia)"]  = inc[total].groupby("anio_envipe").size().values

cuadra = (p1.loc["Suma de las 16 alcaldías"] + p1.loc["Sin alcaldía (999)"] == p1.loc["Total CDMX (ocurrencia)"]).all()
print(f"¿Alcaldías + 999 = total CDMX en los cinco años? {cuadra}\n")
print(p1.to_string())

# Lo que importa para el método: la alcaldía más chica de cada año
filas_alc = p1.iloc[:16]
print("\nMínimo por año:")
for col in filas_alc.columns:
    print(f"  {col}: {filas_alc[col].min():>4}  ({filas_alc[col].idxmin()})")

TABLAS["P1_incidentes_por_alcaldia"] = p1

### UPM por alcaldía

In [ ]:
# Nombres corregidos

# --- Catálogo fijo de nombres (diccionario §4, ENVIPE 2021) ---
# Mejor fijarlo a mano que depender de cómo lo escribió cada año.
NOMBRE_ALC = pd.Series({
    "002": "Azcapotzalco",        "003": "Coyoacán",            "004": "Cuajimalpa de Morelos",
    "005": "Gustavo A. Madero",   "006": "Iztacalco",           "007": "Iztapalapa",
    "008": "La Magdalena Contreras", "009": "Milpa Alta",       "010": "Álvaro Obregón",
    "011": "Tláhuac",             "012": "Tlalpan",             "013": "Xochimilco",
    "014": "Benito Juárez",       "015": "Cuauhtémoc",          "016": "Miguel Hidalgo",
    "017": "Venustiano Carranza",
})
# Control: las claves del catálogo deben ser exactamente las que aparecen en los datos
assert set(NOMBRE_ALC.index) == set(inc["alc_ocu"].dropna()) == set(inc["alc_res"].dropna()), \
       "El catálogo no coincide con las claves de los datos"

# Reetiquetar la tabla de la pregunta 1 (las filas de cierre no empiezan con clave y se quedan igual)
p1.index = [f"{i[:3]} {NOMBRE_ALC[i[:3]]}" if i[:3] in NOMBRE_ALC.index else i for i in p1.index]
TABLAS["P1_incidentes_por_alcaldia"] = p1

# --- Pregunta 2: UPM distintas que aportan incidentes a cada alcaldía ---
# Mismo universo que la pregunta 1. "nunique" cuenta valores distintos de UPM_DIS.
en_alcaldia = universo & inc["alc_ocu"].notna()
p2 = tabla_16x5(inc["UPM_DIS"], en_alcaldia, agregacion="nunique")

# Fila de cierre: UPM distintas en TODA la CDMX (ocurrencia, sin vandalismo).
# Ojo: NO es la suma de las 16 filas. Si una UPM aporta a dos alcaldías,
# cuenta una vez en cada fila pero una sola vez aquí. Esa diferencia es la Parte 3A.
p2.loc["Suma de las 16 filas"] = p2.sum()
p2.loc["UPM distintas, CDMX"]  = (inc[en_alcaldia].groupby("anio_envipe")["UPM_DIS"].nunique().values)

print(p2.to_string())

# --- ¿Hace falta el suavizamiento de You y Hidiroglou? ---
filas_alc = p2.iloc[:16]
print("\nAlcaldía con menos UPM, por año:")
for col in filas_alc.columns:
    print(f"  {col}: {filas_alc[col].min():>3} UPM  ({filas_alc[col].idxmin()})")

for umbral in [1, 5, 10]:
    celdas = (filas_alc <= umbral).sum().sum()
    print(f"Celdas alcaldía-año con {umbral} UPM o menos: {celdas} de 80")

# Incidentes por UPM: qué tan "amontonada" está la muestra (pista para el efecto de diseño)
inc_por_upm = TABLAS["P1_incidentes_por_alcaldia"].iloc[:16] / filas_alc
print(f"\nIncidentes por UPM dentro de una alcaldía: mediana {np.median(inc_por_upm):.2f}, "
      f"rango {inc_por_upm.min().min():.2f}–{inc_por_upm.max().max():.2f}")

TABLAS["P2_upm_por_alcaldia"] = p2

No hay ninguna celda con una sola UPM. El mínimo es Milpa Alta en la ENVIPE 2024, con 7 UPM, y es la única celda de las 80 con 10 o menos. Así que, para d, la condición literal de Santillana no se cumple: la varianza directa se puede calcular en todas las celdas.

Esto vale para d, no necesariamente para k. El denominador de k son solo los incidentes denunciados, que son más o menos uno de cada nueve. Las UPM que aportan denuncias van a ser muchas menos, y ahí sí puede haber celdas con 0 o 1 UPM. 
7 UPM sí permite calcular la varianza, pero la deja inestable. Con 7 conglomerados la varianza tiene unos 6 grados de libertad, así que es una estimación muy ruidosa de sí misma. El suavizamiento de You y Hidiroglou se justifica por eso aunque no haya celdas con una sola UPM, y Santillana lo aplica a todos los municipios, no solo a los de una UPM. Para el escrito, el argumento es "varianzas directas inestables", no "varianzas incalculables".

## Linealización de Taylor

Es la manera de calcular el error estándar de d y k respetando cómo se seleccionó la muestra de la ENVIPE.  d no es un promedio simple, es una razón de dos totales ponderados:

d = Y / X, con Y = Σ w·denuncio (delitos denunciados) y X = Σ w·1 (delitos totales)

Para un total ponderado, la estadística de encuestas sabe calcular la varianza bajo un diseño con estratos y conglomerados. Para una razón no hay una fórmula exacta, porque dividir dos cantidades aleatorias no es una operación lineal.

La expansión en serie de Taylor permite aproximar cualquier función suave por una recta cerca de un punto. Aplicada a la razón, el primer término dice:

d̂ − d ≈ (Ŷ − d·X̂) / X̂

El lado derecho ya no es una razón: es un total de cantidades por incidente,

z = w·(denuncio − d·1) / X̂

Eso es exactamente esta línea del código:

python
z = (y - razon * x) / x.sum()

Cada incidente recibe un z. Si se denunció, su z es positivo, porque empuja a d hacia arriba. Si no se denunció, su z es negativo. La varianza de d̂ es, aproximadamente, la varianza del total de las z. Convertimos un problema difícil, la varianza de una razón, en uno conocido, la varianza de un total.

La varianza de ese total, respetando el diseño. La ENVIPE no sorteó incidentes uno por uno. Dentro de cada estrato (EST_DIS) sorteó UPM (UPM_DIS), y dentro de cada UPM, viviendas. Los incidentes de una misma UPM se parecen entre sí, porque son vecinos, viven la misma inseguridad y tienen la misma agencia del MP cerca. Por eso no se puede tratar cada incidente como independiente. La fórmula trabaja con la UPM como unidad:

Suma las z dentro de cada UPM → z.groupby([EST_DIS, UPM_DIS]).sum()
Agrega las UPM sin incidentes del dominio, con z = 0 → .reindex(upms, fill_value=0.0)
Dentro de cada estrato, mide qué tanto varían los totales de las UPM entre sí → la suma de cuadrados
Suma esa variación sobre todos los estratos, con la corrección n_h/(n_h − 1)

Una UPM sin incidentes en Iztapalapa también es información: dice que en esa parte de la muestra no hubo casos. Quitarla haría que la estimación pareciera más estable de lo que es. Esto es lo que llamamos "estimación por dominio sin filtrar la base".

Por qué ese método y no otro. Las alternativas son los métodos de réplica: jackknife, bootstrap o réplicas repetidas balanceadas. Esos métodos recalculan la estimación muchas veces quitando o remuestreando UPM. Taylor es el que usa el INEGI. 

Varianza por linealización de Taylor de primer orden, con estratos y conglomerados del diseño (Wolter, 2007). 

### Efecto de diseño por alcaldía

Hay que correrlo 160 veces (16 alcaldías × 5 años × 2 parámetros). Anteriormente calculamos la varianza de cada estrato con un apply, que es lento. Así que haremos una versión vectorizada.

In [ ]:
import time
ANIOS = sorted(inc["anio_envipe"].unique())

# Lista completa de UPM de cada año (incluidas las que no tienen incidentes), una sola vez
UPMS = {a: pd.MultiIndex.from_frame(
              per_arch.loc[per_arch["anio_envipe"] == a, ["EST_DIS", "UPM_DIS"]].drop_duplicates())
        for a in ANIOS}


def razon_rapida(inc_a, upms, num, den, dominio):
    """Misma linealización de Taylor que razon_con_error, sin .apply.
    Usa la identidad  Σ(z - z̄)² = Σz² - (Σz)²/n  para que todo sean sumas por grupo,
    que pandas calcula de golpe en lugar de estrato por estrato."""
    w = inc_a["FAC_DEL"]
    y = w * inc_a[num] * dominio
    x = w * inc_a[den] * dominio
    if x.sum() == 0:                       # p. ej. k en una alcaldía sin denuncias: no estimable
        return np.nan, np.nan
    razon = y.sum() / x.sum()
    z = (y - razon * x) / x.sum()

    z_upm = z.groupby([inc_a["EST_DIS"], inc_a["UPM_DIS"]]).sum().reindex(upms, fill_value=0.0)
    por_estrato = z_upm.groupby(level="EST_DIS")
    n_h  = por_estrato.size()
    suma = por_estrato.sum()
    suma_cuadrados = (z_upm ** 2).groupby(level="EST_DIS").sum() - suma ** 2 / n_h
    varianza = (n_h / (n_h - 1) * suma_cuadrados).sum()
    return razon, np.sqrt(varianza)


# --- Validación: la versión rápida contra la original, CDMX, los cinco años ---
print("Validación contra razon_con_error (CDMX, ocurrencia):")
for a in ANIOS:
    inc_a = inc[inc["anio_envipe"] == a].assign(todos=True)
    per_a = per_arch[per_arch["anio_envipe"] == a]
    dom   = ~inc_a["excl_vandalismo"] & inc_a["cdmx_ocu"]
    for param, (num, den) in PARAMETROS.items():
        r0, e0 = razon_con_error(inc_a, per_a,   num, den, dom)
        r1, e1 = razon_rapida(  inc_a, UPMS[a], num, den, dom)
        print(f"  {a} {param}: dif. relativa en EE = {abs(e1 - e0) / e0:.1e}")

# --- Cálculo por alcaldía ---
inicio = time.time()
filas = []
for a in ANIOS:
    inc_a = inc[inc["anio_envipe"] == a].assign(todos=True)
    for clave in NOMBRE_ALC.index:
        dominio = ~inc_a["excl_vandalismo"] & (inc_a["alc_ocu"] == clave)
        for param, (num, den) in PARAMETROS.items():
            est, ee = razon_rapida(inc_a, UPMS[a], num, den, dominio)

            # Incidentes que forman el denominador: todos (d) o solo los denunciados (k)
            base = dominio & inc_a[den]
            n = int(base.sum())
            w = inc_a.loc[base, "FAC_DEL"]

            # Varianza que tendría un muestreo aleatorio simple del mismo tamaño
            v_mas = est * (1 - est) / n if n > 0 else np.nan
            deff  = ee ** 2 / v_mas if (n > 0 and 0 < est < 1) else np.nan
            # Parte del deff que viene solo de la desigualdad de los pesos (fórmula de Kish)
            deff_kish = n * (w ** 2).sum() / w.sum() ** 2 if n > 0 else np.nan

            filas.append({"anio": a, "alc": clave, "param": param, "n": n,
                          "est": est, "ee": ee,
                          "cv": ee / est if est > 0 else np.nan,
                          "deff": deff, "deff_kish": deff_kish,
                          "n_efectiva": n / deff if deff > 0 else np.nan})

diag = pd.DataFrame(filas)       # lo guardamos: la pregunta 4 lo reutiliza
print(f"\n{len(diag)} estimaciones en {time.time() - inicio:.0f} s")


def a_16x5(columna, param):
    """Pasa una columna de `diag` al formato alcaldía × año."""
    t = (diag[diag["param"] == param]
         .pivot(index="alc", columns="anio", values=columna)
         .reindex(NOMBRE_ALC.index))
    t.index   = [f"{c} {NOMBRE_ALC[c]}" for c in t.index]
    t.columns = [ETIQUETA_ANIO[a] for a in t.columns]
    return t


for param in ["d", "k"]:
    t_deff = a_16x5("deff", param)
    sub = diag[diag["param"] == param]
    print(f"\n===== Efecto de diseño de {param} =====")
    print(t_deff.round(2).to_string(na_rep="—"))
    print(f"Mediana: {sub['deff'].median():.2f} · celdas con deff > 2: "
          f"{(sub['deff'] > 2).sum()} de {sub['deff'].notna().sum()}")
    print(f"Descomposición (medianas): pesos (Kish) {sub['deff_kish'].median():.2f} × "
          f"conglomerado y estratos {(sub['deff'] / sub['deff_kish']).median():.2f}")
    TABLAS[f"P3_deff_{param}"] = t_deff.round(2)

# De paso, el CV directo: lo que el escrito necesita para decir "la estimación directa no alcanza"
for param in ["d", "k"]:
    t_cv = 100 * a_16x5("cv", param)
    print(f"\n===== CV directo de {param} (%) =====")
    print(t_cv.round(1).to_string(na_rep="—"))
    TABLAS[f"P3_cv_directo_{param}"] = t_cv.round(1)

El efecto de diseño es moderado, no alto. La mediana es 1.5 para d y k, y casi todo viene de la desigualdad de los pesos (Kish ≈ 1.5). El agrupamiento en UPM aporta casi nada (≈ 1.04), lo cual tiene sentido con unos 2 incidentes por UPM. La muestra efectiva es del orden de dos tercios de la nominal, y en una cuarta parte de las celdas cae por debajo de la mitad. 

31 de 79 celdas de d tienen CV de 30% o más, el umbral del INEGI para precisión baja. Milpa Alta llega a 82.8% en 2024 y Cuajimalpa supera el 44% en tres años.

Hay dos CV de k en 0.0%, y son señales de alarma, no de precisión. Son La Magdalena Contreras 2025 y Milpa Alta 2024. Un CV de cero pasa cuando todas las pocas denuncias tuvieron carpeta, así que k = 100% y la varianza estimada es cero. Lo mismo pasa con el "—" de d en Milpa Alta 2023: cero denuncias, d = 0. Para el Fay-Herriot esto es grave, porque el modelo trata la varianza de muestreo como conocida. Con una varianza de cero, el modelo confiaría por completo en una estimación hecha con tres o cuatro casos. Este es el mejor argumento para el **suavizamiento de You y Hidiroglou**, mejor que el de las UPM.

Gustavo A. Madero 2025 es un caso atípico que hay que revisar: deff de 8.77 en d y 6.54 en k, con un CV de d que salta de alrededor de 20% a 53%. Probablemente hay un incidente con un factor enorme, o varias denuncias concentradas en una UPM. 

In [ ]:
# ¿cuántos incidentes sobreviven al filtro de k?

denunciados = universo & inc["denuncio"]          # el denominador de k
en_alc_ocu  = inc["alc_ocu"].notna()

# --- Denuncias por alcaldía y año (ocurrencia) ---
p4 = tabla_16x5(uno, denunciados & en_alc_ocu)

# Control: debe coincidir con la n que usó la celda 5 para k
assert np.array_equal(p4.values, a_16x5("n", "k").values.astype(int)), "No coincide con diag"

p4["Cinco años"] = p4.sum(axis=1)                 # lo que el Rao-Yu puede aprovechar
p4.loc["Suma de las 16 alcaldías"] = p4.sum()
print("===== Incidentes denunciados (denominador de k), por ocurrencia =====")
print(p4.to_string())

# --- Umbrales: cuántas celdas quedan cortas, en las dos geografías ---
p4_res = tabla_16x5(uno, denunciados & inc["alc_res"].notna(), geografia="alc_res")
print("\nCeldas alcaldía-año (de 80) por debajo de cada umbral de denuncias:")
print(f"  {'umbral':<10}{'ocurrencia':>12}{'residencia':>12}")
for umbral in [1, 5, 10, 15, 30]:
    oc  = (p4.iloc[:16, :5] < umbral).sum().sum()
    res = (p4_res < umbral).sum().sum()
    etiqueta = "= 0" if umbral == 1 else f"< {umbral}"
    print(f"  {etiqueta:<10}{oc:>12}{res:>12}")

# --- Celdas donde k sale degenerado (0% o 100%): varianza directa = 0 ---
carpetas = tabla_16x5(uno, universo & inc["carpeta"] & en_alc_ocu)
denun    = p4.iloc[:16, :5]
degenerado = (denun > 0) & ((carpetas == denun) | (carpetas == 0))
print(f"\nCeldas con k = 0% o k = 100% (EE estimado de cero): {degenerado.sum().sum()}")
for alc, fila in degenerado.iterrows():
    for anio, es in fila.items():
        if es:
            print(f"  {alc}, {anio}: {carpetas.loc[alc, anio]} carpetas de {denun.loc[alc, anio]} denuncias")

# --- UPM que aportan al menos una denuncia (el equivalente de la pregunta 2, para k) ---
p4_upm = tabla_16x5(inc["UPM_DIS"], denunciados & en_alc_ocu, agregacion="nunique")
print("\n===== UPM con al menos una denuncia, por ocurrencia =====")
print(p4_upm.to_string())
for umbral in [1, 3, 5]:
    print(f"Celdas con {umbral} UPM o menos: {(p4_upm <= umbral).sum().sum()} de 80")

TABLAS["P4_denuncias_ocurrencia"]  = p4
TABLAS["P4_denuncias_residencia"]  = p4_res
TABLAS["P4_upm_con_denuncia"]      = p4_upm

# ============================================================
# Revisión: ¿por qué Gustavo A. Madero 2025 tiene deff de 8.8?
# ============================================================
# Hipótesis: un incidente con un factor enorme. FAC_DEL se multiplica por veces/5
# cuando la víctima reporta más de 5 veces el mismo delito (diccionario §6),
# así que alguien con, digamos, 60 extorsiones pesa 12 veces más.

# Peso del incidente más grande dentro de cada celda alcaldía-año
base = inc[universo & en_alc_ocu]
peso_max = (base.groupby(["anio_envipe", "alc_ocu"])["FAC_DEL"]
                .agg(lambda w: w.max() / w.sum())
                .sort_values(ascending=False))
print("\nCeldas donde UN solo incidente concentra más peso (% de los delitos expandidos de la celda):")
for (anio, alc), parte in peso_max.head(6).items():
    print(f"  {alc} {NOMBRE_ALC[alc]:<24} {anio}: {100 * parte:5.1f}%")
print(f"  Mediana de las 80 celdas: {100 * peso_max.median():.1f}%")

gam = base[(base["anio_envipe"] == 2025) & (base["alc_ocu"] == "005")]
print("\nGustavo A. Madero 2025, los 5 incidentes de mayor peso:")
print(gam.nlargest(5, "FAC_DEL")[["BPCOD", "veces", "TD_TIPO", "FAC_ELE", "FAC_DEL",
                                 "denuncio", "carpeta", "UPM_DIS"]]
         .to_string(index=False, float_format=lambda x: f"{x:,.0f}"))

- 22 de 80 celdas tienen menos de 15 denuncias por ocurrencia, y 21 por residencia. El problema no depende de la geografía elegida. Con el umbral de 30, quedan cortas 53 y 55 celdas, dos tercios del panel.
- Milpa Alta tiene 0 denuncias en 2023 y 2 en 2024. Ahí k simplemente no existe como estimación directa.
- La condición literal de Santillana sí aparece en k. Milpa Alta tiene 0 UPM con denuncia en 2023 y una sola en 2024. Para d el suavizamiento de You y Hidiroglou se justificaba por inestabilidad; para k se justifica además porque hay celdas sin varianza calculable.
- Juntar los cinco años ayuda, pero no rescata a todas las alcaldías. Milpa Alta suma 25 denuncias en cinco años, Cuajimalpa 46 y La Magdalena Contreras 57. En esas tres, la estimación del Rao-Yu va a apoyarse sobre todo en la parte del modelo que toma información de otras alcaldías (las covariables), no de otros años. 

Gustavo A. Madero 2025: no es un incidente, es una persona

Mi hipótesis estaba mal planteada. No hay un incidente enorme. Hay una sola víctima de hostigamiento o abuso sexual (BPCOD 13) que reportó 15 eventos. El módulo capta 5 de ellos (TD_TIPO = 5) y el factor se multiplica por 15/5 = 3, así que cada uno pesa 22,682. Son cinco filas idénticas, de la misma persona y la misma UPM, ninguna denunciada.

Mi diagnóstico de "peso máximo por incidente" no lo detectó porque el peso está repartido en cinco filas. Pero para la varianza, lo que cuenta es la UPM. Juntas, esas cinco filas son un bloque de unos 113 mil delitos expandidos que entra o sale de la muestra completo. Eso explica el deff de 8.8.

No es un error de datos. El deff alto refleja una realidad: la estimación de GAM 2025 depende mucho de una sola persona. Pero hay que medirlo bien, con la concentración por persona y por UPM en vez de por incidente, y ver si hay más casos así.

In [ ]:
# ¿Cuánto pesa la persona (o la UPM) más pesada de cada celda?

base = inc[universo & en_alc_ocu].copy()
base["persona"] = base[LLAVE_PERSONA].agg("_".join, axis=1)    # identifica a la víctima dentro del año


def concentracion(nivel):
    """Para cada celda alcaldía-año: fracción de los delitos expandidos que aporta
    la unidad (persona o UPM) más pesada."""
    peso = base.groupby(["anio_envipe", "alc_ocu", nivel])["FAC_DEL"].sum()
    total_celda = peso.groupby(level=[0, 1]).transform("sum")   # total de la celda, repetido en cada fila
    return (peso / total_celda).groupby(level=[0, 1]).max()


conc = pd.DataFrame({
    "persona más pesada (%)": 100 * concentracion("persona"),
    "UPM más pesada (%)":     100 * concentracion("UPM_DIS"),
})
deff_d = diag[diag["param"] == "d"].set_index(["anio", "alc"])["deff"]
deff_d.index.names = conc.index.names                           # mismos nombres para que se alineen
conc["deff d"] = deff_d

conc = conc.reset_index()
conc["alcaldía"] = conc["alc_ocu"].map(NOMBRE_ALC)
cols = ["anio_envipe", "alcaldía", "persona más pesada (%)", "UPM más pesada (%)", "deff d"]

print("Las 8 celdas con mayor deff de d:")
print(conc.nlargest(8, "deff d")[cols].to_string(index=False, float_format=lambda x: f"{x:.1f}"))

print(f"\nMedianas de las 80 celdas: persona {conc['persona más pesada (%)'].median():.1f}% · "
      f"UPM {conc['UPM más pesada (%)'].median():.1f}%")

# ¿La concentración explica el deff? Correlación de rangos (Spearman), sin suponer una relación lineal
for col in ["persona más pesada (%)", "UPM más pesada (%)"]:
    rho = conc[[col, "deff d"]].corr(method="spearman").iloc[0, 1]
    print(f"Spearman entre deff de d y {col}: {rho:.2f}")

# Cuántas celdas dependen de una sola persona en más de 10% o 20%
for umbral in [10, 20]:
    n = (conc["persona más pesada (%)"] > umbral).sum()
    print(f"Celdas donde una sola persona aporta más de {umbral}% de los delitos: {n} de 80")

- Gustavo A. Madero 2025 es un caso extremo y aislado. Una sola víctima aporta el 35.5% de los delitos expandidos de su celda. Ninguna otra celda tiene un deff arriba de 4, y solo 4 de 80 celdas tienen una persona que aporte más del 20%.
- En general, la concentración no explica el efecto de diseño (Spearman ≈ 0). Esto confirma la descomposición: la pérdida típica de precisión viene de la desigualdad de los pesos, no de víctimas con muchos incidentes ni del agrupamiento geográfico. GAM 2025 es la excepción, no el patrón.
- 19 de 80 celdas dependen en más de 10% de una sola persona. Esto esuna limitación: es la fragilidad típica de las estimaciones directas con muestras chicas, y es lo que el Fay-Herriot corrige. Al suavizar la varianza, la celda de GAM 2025 recibirá menos peso directo y más del modelo.

### Razones de no denuncia

In [ ]:
RAZONES = {"01": "Miedo al agresor",          "02": "Miedo a extorsión",
           "03": "Delito de poca importancia", "04": "Pérdida de tiempo",
           "05": "Trámites largos y difíciles","06": "Desconfianza en la autoridad",
           "07": "No tenía pruebas",           "08": "Actitud hostil de la autoridad",
           "09": "Otra",                       "99": "NS/NR"}
AUTORIDAD = ["02", "04", "05", "06", "08"]       # ⚠️ confirmar contra la nota del boletín

# Regiones del INEGI, definición 2023+ aplicada a los cinco años (diccionario §5)
REGIONES = {"Norte":    ["005", "006", "015", "017"],
            "Sur":      ["003", "008", "012", "014"],
            "Oriente":  ["007", "009", "011", "013"],
            "Poniente": ["002", "004", "010", "016"]}

# --- Chequeos: BP1_23 debe estar llena en TODOS los no denunciados y vacía en los denunciados ---
no_den = universo & inc["cdmx_ocu"] & ~inc["denuncio"]
print(f"No denunciados CDMX con BP1_23 vacía (debe ser 0): {inc.loc[no_den, 'BP1_23'].isna().sum()}")
print(f"Denunciados CDMX con BP1_23 llena (debe ser 0):    "
      f"{inc.loc[universo & inc['cdmx_ocu'] & inc['denuncio'], 'BP1_23'].notna().sum()}")
fuera = set(inc.loc[no_den, "BP1_23"].dropna()) - set(RAZONES)
print(f"Códigos fuera del catálogo: {fuera if fuera else 'ninguno'}\n")

# --- Una columna verdadero/falso por razón (lo que razon_rapida necesita como numerador) ---
inc6 = inc.assign(no_denuncio=~inc["denuncio"])
VARIABLES = {}
for codigo, nombre in RAZONES.items():
    inc6[f"razon_{codigo}"] = inc6["BP1_23"] == codigo
    VARIABLES[f"razon_{codigo}"] = f"{codigo} {nombre}"
inc6["razon_autoridad"] = inc6["BP1_23"].isin(AUTORIDAD)
VARIABLES["razon_autoridad"] = "Atribuibles a la autoridad (02, 04, 05, 06, 08)"

# --- Estimación: proporción entre los NO denunciados, con su error estándar ---
GRUPOS = {"CDMX": None, **REGIONES}     # None = toda la ciudad, incluidos los 999
filas = []
for a in ANIOS:
    inc_a = inc6[inc6["anio_envipe"] == a]
    univ_a = ~inc_a["excl_vandalismo"]
    for grupo, claves in GRUPOS.items():
        dom = univ_a & (inc_a["cdmx_ocu"] if claves is None else inc_a["alc_ocu"].isin(claves))
        n = int((dom & inc_a["no_denuncio"]).sum())
        for var, etiqueta in VARIABLES.items():
            est, ee = razon_rapida(inc_a, UPMS[a], var, "no_denuncio", dom)
            filas.append({"anio": a, "grupo": grupo, "razon": etiqueta, "est": est, "ee": ee, "n": n})
r6 = pd.DataFrame(filas)
ORDEN = list(VARIABLES.values())

# --- Tabla 1: CDMX por año, más el promedio de los cinco años ---
cdmx = r6[r6["grupo"] == "CDMX"]
t6 = 100 * cdmx.pivot(index="razon", columns="anio", values="est").reindex(ORDEN)
t6.columns = [ETIQUETA_ANIO[a] for a in t6.columns]
prom = cdmx.groupby("razon").agg(est=("est", "mean"),
                                 ee=("ee", lambda e: np.sqrt((e ** 2).sum()) / len(e)))
t6["Promedio 5 años"] = 100 * prom["est"].reindex(ORDEN)
t6["CV promedio (%)"] = 100 * (prom["ee"] / prom["est"]).reindex(ORDEN)

print("===== Razones de no denuncia, CDMX (% de los delitos no denunciados) =====")
print(f"Incidentes no denunciados por año: {cdmx.groupby('anio')['n'].first().to_dict()}")
print(t6.round(1).to_string())

# --- Tabla 2: por región, promedio de los cinco años ---
reg = r6[r6["grupo"] != "CDMX"].groupby(["grupo", "razon"]).agg(
        est=("est", "mean"), ee=("ee", lambda e: np.sqrt((e ** 2).sum()) / len(e)))
t6r = 100 * reg["est"].unstack("grupo").reindex(ORDEN)[list(REGIONES)]
cv6r = 100 * (reg["ee"] / reg["est"]).unstack("grupo").reindex(ORDEN)[list(REGIONES)]

print("\n===== Por región, promedio de los cinco años (%) =====")
print(f"Incidentes no denunciados, cinco años: "
      f"{r6[r6['grupo'] != 'CDMX'].groupby(['grupo', 'anio'])['n'].first().groupby('grupo').sum().to_dict()}")
print(t6r.round(1).to_string())
print("\nCeldas región × razón con CV > 30% (leer con cautela):")
for razon, fila in (cv6r > 30).iterrows():
    malas = [g for g, es in fila.items() if es]
    if malas:
        print(f"  {razon}: {', '.join(malas)}")

TABLAS["P6_razones_cdmx"]   = t6.round(1)
TABLAS["P6_razones_region"] = t6r.round(1)
TABLAS["P6_cv_region"]      = cv6r.round(1)

- Siete de cada diez delitos no denunciados en la CDMX se explican por causas atribuibles a la autoridad, y la cifra es muy estable: entre 67.6% y 75.4% según el año, con un CV de 1.1% en el promedio. La principal es "pérdida de tiempo" (38%), seguida de desconfianza (17%) y trámites largos (11%). El miedo al agresor (4.5%) y la falta de pruebas (8%) pesan poco.
- Las regiones casi no se distinguen. "Atribuibles a la autoridad" va de 69.1% (Sur) a 72.0% (Norte), y "pérdida de tiempo" de 35.6% a 40.3%. Son diferencias de pocos puntos, del orden del ruido muestral. No las presentaría como contrastes regionales.
- Esto tiene una implicación interesante para tu argumento. La composición de las razones es homogénea en el territorio. Así que, si d varía entre alcaldías, no parece ser porque en unas se desconfíe y en otras se ahorre tiempo: la barrera ciudadana tiene la misma forma en toda la ciudad y cambia de intensidad. Para el tomador de decisiones, eso apunta a que la intervención sobre d es de alcance de ciudad (reducir el costo de denunciar: tiempo y trámites), mientras que la palanca territorial estaría en k. Es una lectura tentativa: aquí solo podemos verla a nivel de región.
- Una tendencia a anotar: "delito de poca importancia" sube de 6.1% a 10.1% y 9.5% en las dos ediciones más recientes. Puede ser un cambio en la mezcla de delitos, por ejemplo más fraudes de monto bajo. No lo exploraría hoy, pero vale la pena registrarlo.

[Confirmar en el boletín la agrupación de "atribuibles a la autoridad"]

In [ ]:
# ---------- Parte A: ¿dónde están las variables RESUL_* y qué valores traen? ----------
# Solo leemos los encabezados (nrows=0) y luego únicamente esas columnas: es rápido.
print("===== Parte A: variables de resultado de la entrevista =====")
for anio, carpeta in CARPETAS.items():
    for prefijo in ["TVivienda", "THogar", "TPer_Vic1", "TPer_Vic2"]:
        ruta = buscar_archivo(carpeta, prefijo)
        columnas = pd.read_csv(ruta, nrows=0, encoding=CODIFICACION).columns
        resul = [c for c in columnas if c.upper().startswith("RESUL")]
        if not resul:
            continue
        valores = pd.read_csv(ruta, usecols=resul, dtype=str, encoding=CODIFICACION)
        for c in resul:
            conteo = valores[c].value_counts(dropna=False).sort_index().to_dict()
            print(f"  {anio} {prefijo:<10} {c:<8} {conteo}")

# ---------- Parte B: BP1_24 sin dato entre los denunciados ----------
print("\n===== Parte B: ¿se inició carpeta? (BP1_24) entre los incidentes denunciados =====")
den_ocu = universo & inc["denuncio"] & en_alc_ocu
b = inc[den_ocu].copy()
b["sin_dato"] = b["BP1_24"].isna() | (b["BP1_24"] == "9")

# ¿Los vacíos son los que ya conocemos? (denunció otro integrante y la víctima no conoció el proceso)
vacios = b["BP1_24"].isna()
explicados = vacios & (b["BP1_21"] == "1") & (b["BP1_22"] == "2")
print(f"BP1_24 vacía: {vacios.sum()} · de ellas, con BP1_21 = 1 y BP1_22 = 2: {explicados.sum()}")
print(f"BP1_24 = 9 (NS/NR): {(b['BP1_24'] == '9').sum()}")

# Por año, CDMX (alcaldías válidas)
por_anio = b.groupby("anio_envipe").agg(denuncias=("sin_dato", "size"),
                                         sin_dato=("sin_dato", "sum"))
por_anio["% sin dato"] = 100 * por_anio["sin_dato"] / por_anio["denuncias"]
por_anio.index = [ETIQUETA_ANIO[a] for a in por_anio.index]
print("\nPor año:")
print(por_anio.round(1).to_string())

# Por alcaldía, cinco años juntos (un año solo da conteos de 0 a 3)
w = b["FAC_DEL"]
por_alc = b.assign(w_carpeta=w * b["carpeta"], w_con_dato=w * ~b["sin_dato"]).groupby("alc_ocu").agg(
    denuncias=("sin_dato", "size"), sin_dato=("sin_dato", "sum"),
    w_den=("FAC_DEL", "sum"), w_carpeta=("w_carpeta", "sum"), w_con_dato=("w_con_dato", "sum"))
por_alc["% sin dato"] = 100 * por_alc["sin_dato"] / por_alc["denuncias"]
# Sensibilidad: k con la convención del INEGI (sin dato = sin carpeta) vs. excluyendo los sin dato
por_alc["k convención (%)"]     = 100 * por_alc["w_carpeta"] / por_alc["w_den"]
por_alc["k sin los faltantes (%)"] = 100 * por_alc["w_carpeta"] / por_alc["w_con_dato"]
por_alc["diferencia (pp)"] = por_alc["k sin los faltantes (%)"] - por_alc["k convención (%)"]
por_alc = por_alc.reindex(NOMBRE_ALC.index)
por_alc.index = [f"{c} {NOMBRE_ALC[c]}" for c in por_alc.index]

columnas_p7 = ["denuncias", "sin_dato", "% sin dato", "k convención (%)",
               "k sin los faltantes (%)", "diferencia (pp)"]
print("\nPor alcaldía, cinco años juntos (k ponderado con FAC_DEL, descriptivo):")
print(por_alc[columnas_p7].round(1).to_string())

# ¿La no respuesta es homogénea entre alcaldías? Prueba ji cuadrada de independencia
try:
    from scipy.stats import chi2_contingency
    tabla_cont = np.column_stack([por_alc["sin_dato"], por_alc["denuncias"] - por_alc["sin_dato"]])
    chi2, p, gl, esperados = chi2_contingency(tabla_cont)
    print(f"\nJi cuadrada: {chi2:.1f}, gl = {gl}, p = {p:.3f} · "
          f"celdas con frecuencia esperada < 5: {(esperados < 5).sum()} de {esperados.size}")
except ImportError:
    print("\n(scipy no está instalado: se omite la prueba)")

TABLAS["P7_bp1_24_por_anio"]     = por_anio.round(1)
TABLAS["P7_bp1_24_por_alcaldia"] = por_alc[columnas_p7].round(1)

La ji cuadrada da p = 0.59: no hay evidencia de que la no respuesta en BP1_24 se concentre en algunas alcaldías. Hay 4 de 32 celdas con frecuencia esperada menor que 5, que está por debajo del 20% que suele tolerarse, así que la aproximación es aceptable. 

La no respuesta en el inicio de carpeta (5–9% de las denuncias según el año) no difiere entre alcaldías (χ² = 13.1, gl = 15, p = 0.59). La convención del INEGI de contarla como 'sin carpeta' reduce el nivel de k entre 0 y 6.3 puntos, de manera aproximadamente uniforme.

In [ ]:
# ¿La no respuesta en BP1_24 difiere entre alcaldías?

rng = np.random.default_rng(20261008)      # semilla fija: el resultado se puede reproducir

n_alc    = por_alc["denuncias"].to_numpy()
faltan   = por_alc["sin_dato"].to_numpy()
p_global = faltan.sum() / n_alc.sum()
esperado = n_alc * p_global                # faltantes esperados si todas tuvieran la misma tasa


def estadistico_ji2(x):
    """Ji cuadrada de una tabla 16 × 2 (con dato / sin dato), con los esperados fijos."""
    return (((x - esperado) ** 2) / esperado
            + (((n_alc - x) - (n_alc - esperado)) ** 2) / (n_alc - esperado)).sum()


observado = estadistico_ji2(faltan)

# Permutación: repartimos al azar las MISMAS etiquetas "sin dato" entre todas las denuncias,
# respetando cuántas denuncias tiene cada alcaldía. Así se ve el estadístico cuando la
# no respuesta NO depende de la alcaldía.
alcaldia_de_cada_denuncia = np.repeat(np.arange(16), n_alc)
etiquetas = np.r_[np.ones(faltan.sum()), np.zeros(n_alc.sum() - faltan.sum())]

B = 10_000
simulados = np.empty(B)
for i in range(B):
    revueltas = rng.permutation(etiquetas)
    simulados[i] = estadistico_ji2(np.bincount(alcaldia_de_cada_denuncia, weights=revueltas, minlength=16))

p_valor = (1 + (simulados >= observado).sum()) / (1 + B)
print(f"Ji cuadrada observada: {observado:.1f} · p de permutación ({B:,} réplicas): {p_valor:.3f}")
print(f"Celdas con esperado < 5: {(esperado < 5).sum()} (por eso la permutación y no la tabla de ji cuadrada)")

# ¿La convención cambia el ORDEN de las alcaldías en k?
rho = por_alc[["k convención (%)", "k sin los faltantes (%)"]].corr(method="spearman").iloc[0, 1]
print(f"\nSpearman entre k con convención y k sin faltantes: {rho:.3f}")
cambio_rango = (por_alc["k convención (%)"].rank(ascending=False)
                - por_alc["k sin los faltantes (%)"].rank(ascending=False)).abs()
print(f"Alcaldías que cambian de lugar: {(cambio_rango > 0).sum()} de 16 · "
      f"cambio máximo: {cambio_rango.max():.0f} lugares")

La permutación da p = 0.598, prácticamente igual que la ji cuadrada clásica (0.594). La conclusión es sólida: no hay no respuesta diferencial entre alcaldías.

Spearman da 0.90, 13 de las 16 alcaldías cambian de lugar y una se mueve 6 posiciones. Pasa porque los k están muy apretados, casi todos entre 61% y 74%, y en un rango tan estrecho bastan unos puntos para reordenar. La convención del INEGI reduce k entre 0 y 6.3 puntos y conserva el orden de manera aproximada (ρ = 0.90); se reporta como principal por comparabilidad con las cifras oficiales y la versión sin faltantes como robustez. Es el mismo tipo de ρ que reporta Buil-Gil.

### Microdato de carpetas FGJ CDMX

In [ ]:
# explorar y convertir a parquet

RUTA_FGJ = (BASE.parent / "Carpetas de Investigación de la FGJ"
            / "carpetasFGJ_acumulado_2025_01.csv")
print(f"¿Existe el archivo? {RUTA_FGJ.exists()}")

# --- 1. Solo encabezados y 3 filas de muestra (instantáneo) ---
# Probamos UTF-8; si falla, latin-1. Guardamos la que funcionó para la lectura completa.
for codif in ["utf-8", "latin-1"]:
    try:
        muestra = pd.read_csv(RUTA_FGJ, nrows=3, dtype=str, encoding=codif)
        CODIF_FGJ = codif
        break
    except UnicodeDecodeError:
        continue
print(f"Codificación: {CODIF_FGJ}\n")
print("Columnas y una fila de ejemplo:")
for col in muestra.columns:
    print(f"  {col:<30} {muestra[col].iloc[0]}")

# --- 2. Elegimos las columnas por su nombre (no las suponemos) ---
claves = ["categ", "delito", "fecha", "anio", "ao_", "alcaldia", "municipio", "fiscalia",
          "competencia", "latitud", "longitud"]
utiles = [c for c in muestra.columns if any(k in c.lower() for k in claves)]
print(f"\nColumnas que vamos a leer: {utiles}")

# --- 3. Lectura completa, solo esas columnas (aquí sí tarda) ---
inicio = time.time()
try:
    fgj = pd.read_csv(RUTA_FGJ, usecols=utiles, dtype=str, encoding=CODIF_FGJ)
except UnicodeDecodeError:          # a veces el problema aparece hasta media lectura
    CODIF_FGJ = "latin-1"
    fgj = pd.read_csv(RUTA_FGJ, usecols=utiles, dtype=str, encoding=CODIF_FGJ)
print(f"{len(fgj):,} filas leídas en {time.time() - inicio:.0f} s (codificación final: {CODIF_FGJ})")

# --- 4. Guardar como parquet: la próxima vez se lee en segundos ---
RUTA_FGJ_PARQUET = CARPETA_SALIDA / "fgj_carpetas.parquet"
fgj.to_parquet(RUTA_FGJ_PARQUET, index=False)
print(f"Guardado: {RUTA_FGJ_PARQUET.name}\n")

# --- 5. Lo que necesito ver ---
for col in utiles:
    distintos = fgj[col].nunique()
    if any(k in col.lower() for k in ["latitud", "longitud"]):
        continue                                                 # coordenadas: no hace falta listarlas
    print(f"--- {col}: {distintos:,} valores distintos · vacíos: {fgj[col].isna().sum():,} ---")
    if "fecha" in col.lower():
        print(f"  ejemplos: {fgj[col].dropna().sample(3, random_state=1).tolist()}")
    elif distintos <= 40:                                        # categorías, alcaldías, años: todo
        print(fgj[col].value_counts(dropna=False).to_string())
    else:                                                        # delito desagregado: los 50 más frecuentes
        print(fgj[col].value_counts().head(50).to_string())
    print()

- 2024 está completo. Las carpetas iniciadas en 2024 (232 mil) son comparables con las de 2023 (242 mil), y 2025 solo trae enero, de ahí el nombre _2025_01. 
- Hay que contar por anio_hecho, no por anio_inicio. La ENVIPE pregunta por delitos ocurridos en el año. Pero eso trae un detalle: un delito de 2024 cuya carpeta se abrió después de enero de 2025 no está en el archivo. Hay que medir cuánto se pierde por ese corte, y la celda de abajo lo estima con los años anteriores.
- categoria_delito no sirve para la correspondencia. "Delito de bajo impacto" se lleva 1.7 de 2.1 millones de carpetas y mete en una sola bolsa fraude, amenazas y robo de objetos. La tabla tiene que hacerse con delito, que trae 357 valores.
- Hay registros que no son delitos: competencia = "HECHO NO DELICTIVO" o "INCOMPETENCIA", y la categoría "HECHO NO DELICTIVO". competencia está vacía en la mitad de los registros, seguramente los de los primeros años, así que hay que ver cómo se combinan las dos variables.
- Las carpetas también tienen su "999": 24,896 con alcaldía vacía y 17,586 con "CDMX (indeterminada)". 
- El delito más frecuente es violencia familiar (261 mil), y no tiene contraparte en la ENVIPE. 

In [ ]:
# Microdato FGJ: fechas, alcaldías y registros que no son delitos

import unicodedata

# --- 1. Fechas: de texto a fecha ---
# format= le dice a pandas el formato exacto (aaaa-mm-dd, como en los ejemplos);
# errors="coerce" convierte en NaT (fecha vacía) lo que no cumpla el formato, en vez de detenerse.
fgj["f_inicio"] = pd.to_datetime(fgj["fecha_inicio"], format="%Y-%m-%d", errors="coerce")
fgj["f_hecho"]  = pd.to_datetime(fgj["fecha_hecho"],  format="%Y-%m-%d", errors="coerce")
print(f"Fechas de inicio que no se pudieron leer: {fgj['f_inicio'].isna().sum() - fgj['fecha_inicio'].isna().sum()}")
print(f"Fechas del hecho que no se pudieron leer: {fgj['f_hecho'].isna().sum() - fgj['fecha_hecho'].isna().sum()}")
CORTE = fgj["f_inicio"].max()
print(f"Fecha de inicio más antigua: {fgj['f_inicio'].min().date()} · más reciente (corte): {CORTE.date()}")

# ¿anio_hecho coincide con el año de fecha_hecho?
anio_col = pd.to_numeric(fgj["anio_hecho"], errors="coerce")
distintos = (anio_col != fgj["f_hecho"].dt.year) & anio_col.notna() & fgj["f_hecho"].notna()
print(f"Registros donde anio_hecho ≠ año de fecha_hecho: {distintos.sum()}")
fgj["anio_h"] = fgj["f_hecho"].dt.year

# --- 2. Truncamiento por el corte ---
# Un delito ocurrido en 2024 cuya carpeta se abrió después del corte no aparece.
# Estimamos cuánto se pierde con los años anteriores: para cada año h, qué fracción
# de sus carpetas se abrió más allá de la misma distancia del fin de año que tiene el corte.
desfase = CORTE - pd.Timestamp("2024-12-31")
print(f"\nTruncamiento esperado para los hechos de 2024 (corte {desfase.days} días después del fin de año):")
for h in range(2019, 2024):
    del_anio = fgj[fgj["anio_h"] == h]
    tardias = (del_anio["f_inicio"] > pd.Timestamp(f"{h}-12-31") + desfase).mean()
    print(f"  hechos de {h}: {100 * tardias:.1f}% de sus carpetas se abrieron después de ese plazo")

# --- 3. Registros que no son delitos ---
print("\nCategoría 'HECHO NO DELICTIVO' contra la variable competencia:")
print(pd.crosstab(fgj["categoria_delito"] == "HECHO NO DELICTIVO",
                  fgj["competencia"].fillna("(vacía)")).to_string())
print("\n¿En qué años de inicio está vacía 'competencia'? (% vacía)")
print((100 * fgj["competencia"].isna().groupby(fgj["anio_inicio"]).mean()).round(1).to_string())

fgj["no_delito"] = (fgj["competencia"].isin(["HECHO NO DELICTIVO", "INCOMPETENCIA"])
                    | (fgj["categoria_delito"] == "HECHO NO DELICTIVO"))
print(f"\nRegistros marcados como no delito: {fgj['no_delito'].sum():,}")

# --- 4. Alcaldía del hecho → misma clave que la ENVIPE ---
def sin_acentos(texto):
    """'Álvaro Obregón' → 'ALVARO OBREGON'. Separa cada letra de su acento y descarta el acento."""
    return unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode().upper()

CLAVE_POR_NOMBRE = {sin_acentos(nombre): clave for clave, nombre in NOMBRE_ALC.items()}
fgj["alc_hecho"] = fgj["alcaldia_hecho"].map(CLAVE_POR_NOMBRE)
print("\nValores de alcaldia_hecho que NO se asignaron a una clave (deben ser solo vacía e indeterminada):")
print(fgj.loc[fgj["alc_hecho"].isna(), "alcaldia_hecho"].value_counts(dropna=False).to_string())

# --- 5. Carpetas por alcaldía y año del hecho, 2020–2024, solo delitos ---
periodo = fgj["anio_h"].between(2020, 2024) & ~fgj["no_delito"]
t_fgj = (fgj[periodo].assign(alc=fgj["alc_hecho"].fillna("sin alcaldía"))
         .pivot_table(index="alc", columns="anio_h", values="delito", aggfunc="size", fill_value=0))
t_fgj.index = [f"{c} {NOMBRE_ALC[c]}" if c in NOMBRE_ALC.index else c for c in t_fgj.index]
t_fgj.loc["Total"] = t_fgj.sum()
print("\nCarpetas (delitos, por año del HECHO):")
print(t_fgj.to_string())
print("\n% sin alcaldía por año:", (100 * t_fgj.loc["sin alcaldía"] / t_fgj.loc["Total"]).round(1).to_dict())

In [ ]:
# Fechas de la FGJ: diagnóstico, lectura robusta y validación

# --- 1. ¿Cómo vienen escritas las fechas que fallaron? ---
for texto, fecha in [("fecha_inicio", "f_inicio"), ("fecha_hecho", "f_hecho")]:
    fallan = fgj[texto].notna() & fgj[fecha].isna()
    print(f"{texto}: {fallan.sum():,} fallaron. Ejemplos:")
    print("  ", fgj.loc[fallan, texto].sample(8, random_state=1).tolist())
    print("   por año de inicio:", fgj.loc[fallan, "anio_inicio"].value_counts().sort_index().to_dict(), "\n")

# --- 2. Columnas de mes (no las habíamos leído): sirven para validar ---
MESES = {"ENERO": 1, "FEBRERO": 2, "MARZO": 3, "ABRIL": 4, "MAYO": 5, "JUNIO": 6, "JULIO": 7,
         "AGOSTO": 8, "SEPTIEMBRE": 9, "OCTUBRE": 10, "NOVIEMBRE": 11, "DICIEMBRE": 12}
meses = pd.read_csv(RUTA_FGJ, usecols=["mes_inicio", "mes_hecho"], dtype=str, encoding=CODIF_FGJ)
assert len(meses) == len(fgj), "El archivo y el parquet no tienen las mismas filas"
for col in ["mes_inicio", "mes_hecho"]:
    limpio = meses[col].str.strip().map(sin_acentos, na_action="ignore")
    no_reconocidos = limpio[limpio.notna() & ~limpio.isin(MESES)].value_counts()
    if len(no_reconocidos):
        print(f"Nombres de mes no reconocidos en {col}:", no_reconocidos.to_dict())
    fgj[f"{col}_n"] = limpio.map(MESES)

# --- 3. Lectura probando formatos en orden ---
FORMATOS = ["%Y-%m-%d", "%Y-%m-%d %H:%M:%S", "%d/%m/%Y", "%d/%m/%Y %H:%M:%S",
            "%d/%m/%Y %H:%M", "%Y/%m/%d", "%m/%d/%Y"]


def leer_fechas(texto):
    """Prueba cada formato solo en las fechas que siguen sin leerse.
    Imprime cuántas atrapó cada formato, para saber qué pasó."""
    resultado = pd.Series(pd.NaT, index=texto.index, dtype="datetime64[ns]")
    texto = texto.str.strip()
    for formato in FORMATOS:
        falta = resultado.isna() & texto.notna()
        if not falta.any():
            break
        intento = pd.to_datetime(texto[falta], format=formato, errors="coerce")
        resultado.loc[falta] = intento
        if intento.notna().sum():
            print(f"    {formato:<20} {intento.notna().sum():>10,}")
    print(f"    {'(sin leer)':<20} {(resultado.isna() & texto.notna()).sum():>10,}")
    return resultado


print("\nfecha_inicio:")
fgj["f_inicio"] = leer_fechas(fgj["fecha_inicio"])
print("fecha_hecho:")
fgj["f_hecho"] = leer_fechas(fgj["fecha_hecho"])

# --- 4. Validación contra las columnas de año y mes del propio archivo ---
anio_ini = pd.to_numeric(fgj["anio_inicio"], errors="coerce")
anio_hec = pd.to_numeric(fgj["anio_hecho"], errors="coerce")
for fecha, anio, mes in [("f_inicio", anio_ini, "mes_inicio_n"), ("f_hecho", anio_hec, "mes_hecho_n")]:
    hay = fgj[fecha].notna() & anio.notna() & fgj[mes].notna()
    mal_anio = (fgj.loc[hay, fecha].dt.year  != anio[hay]).sum()
    mal_mes  = (fgj.loc[hay, fecha].dt.month != fgj.loc[hay, mes]).sum()
    print(f"{fecha}: año distinto {mal_anio:,} · mes distinto {mal_mes:,}  (de {hay.sum():,} comparables)")

# --- 5. Corte y truncamiento, ahora sí ---
CORTE = fgj["f_inicio"].max()
desfase = CORTE - pd.Timestamp("2024-12-31")
print(f"\nCorte: {CORTE.date()} ({desfase.days} días después del fin de 2024)")
for h in range(2019, 2024):
    del_anio = fgj[anio_hec == h]
    tardias = (del_anio["f_inicio"] > pd.Timestamp(f"{h}-12-31") + desfase).mean()
    print(f"  hechos de {h}: {100 * tardias:.2f}% de sus carpetas se abrieron después de ese plazo")

# --- 6. La tabla por alcaldía, con el AÑO DE LA COLUMNA (no de la fecha) ---
fgj["anio_h"] = anio_hec
periodo = fgj["anio_h"].between(2020, 2024) & ~fgj["no_delito"]
t_fgj = (fgj[periodo].assign(alc=fgj["alc_hecho"].fillna("sin alcaldía"))
         .pivot_table(index="alc", columns="anio_h", values="delito", aggfunc="size", fill_value=0))
t_fgj.columns = t_fgj.columns.astype(int)
t_fgj.index = [f"{c} {NOMBRE_ALC[c]}" if c in NOMBRE_ALC.index else c for c in t_fgj.index]
t_fgj.loc["Total"] = t_fgj.sum()
print("\nCarpetas (delitos, por año del hecho):")
print(t_fgj.to_string())
print("\n% sin alcaldía:", (100 * t_fgj.loc["sin alcaldía"] / t_fgj.loc["Total"]).round(1).to_dict())

- El corte real es el 31 de enero de 2025. El total de 2024 sube de 170 mil a 207 mil carpetas. Aun así, a 2024 le falta alrededor de 4–6% de sus carpetas por el rezago. En los años anteriores, entre 4.1% y 5.9% de las carpetas de cada año se abrieron más de un mes después de que terminó el año. Así que la caída de 2023 a 2024 (de 229.7 mil a 207.3 mil, −9.7%) es en parte real y en parte, alrededor de la mitad, un artefacto del corte. Hay dos opciones para el mapa corregido: inflar 2024 por un factor de cerca de 1.05, o reportarlo como cota inferior. 
- Las carpetas sin alcaldía pasan de 0.3–0.6% a 3.1–3.7% en 2023–2024, un cambio de registro en la Fiscalía que va junto al punto 3B.

In [ ]:
# Borrador de correspondencia BPCOD ↔ delito FGJ

import re

BPCOD_NOMBRE = {"01": "Robo total de vehículo",        "02": "Robo de accesorios/autopartes",
                "03": "Vandalismo (excluido)",         "04": "Robo a casa habitación",
                "05": "Robo/asalto en calle o transporte", "06": "Otro tipo de robo",
                "07": "Fraude bancario",               "08": "Fraude al consumidor",
                "07/08": "Fraude (bancario o al consumidor)",
                "09": "Extorsión",                     "10": "Amenazas verbales",
                "11": "Lesiones",                      "12": "Secuestro",
                "13": "Hostigamiento/abuso sexual",    "14": "Violación sexual",
                "15": "Otros delitos",                 "—": "Sin correspondencia en la ENVIPE",
                "?": "Sin asignar (revisar)"}

# (BPCOD, expresión regular sobre el delito SIN acentos y en mayúsculas, confianza, razón)
# Se aplican EN ORDEN y gana la primera que coincide: por eso las exclusiones van primero.
REGLAS = [
    ("03", r"^DANO EN PROPIEDAD AJENA INTENCIONAL", "alta",
     "Vandalismo: excluido del universo, igual que en la ENVIPE"),
    ("—",  r"CULPOS", "alta",
     "Delito culposo (p. ej. de tránsito): no es victimización que capte la ENVIPE"),
    ("—",  r"ROBO A NEGOCIO|FARDEROS|ROBO A TRANSPORTISTA|ROBO A (OFICINA|INSTITUCION|SUCURSAL|EMPRESA|BANCO|LOCAL)", "alta",
     "La víctima es un negocio o institución: lo mide la ENVE, no la ENVIPE"),
    ("—",  r"^DENUNCIA DE HECHOS|^PERDIDA DE LA VIDA", "alta",
     "No es un delito tipificado"),
    ("—",  r"HOMICIDIO|FEMINICIDIO|NARCOMENUDEO|PORTACION", "alta",
     "La ENVIPE no capta este delito"),
    ("01", r"^ROBO DE VEHICULO DE SERVICIO PARTICULAR", "alta", ""),
    ("06", r"^ROBO DE VEHICULO DE PEDALES", "dudosa",
     "Bicicleta: verificar en el cuestionario si la ENVIPE la cuenta en 'otro tipo de robo'"),
    ("01", r"^ROBO DE (MOTOCICLETA|VEHICULO)", "dudosa",
     "Motocicleta, servicio público o de carga: verificar si entra en robo total de vehículo"),
    ("02", r"^ROBO DE ACCESORIOS DE AUTO|AUTOPARTE", "alta", ""),
    ("02", r"^ROBO DE PLACA|^ROBO DE OBJETOS DEL INTERIOR DE UN VEHICULO", "dudosa",
     "Frontera entre 02 (accesorios) y 06 (otro robo)"),
    ("04", r"^ROBO A CASA HABITACION", "alta", ""),
    ("05", r"CONDUCTOR", "dudosa",
     "Robo al conductor en su vehículo: entre 05 y 01 (si se llevan el vehículo)"),
    ("05", r"^ROBO A TRANSEUNTE|^ROBO A PASAJERO", "alta", ""),
    ("05", r"^ROBO A REPARTIDOR|^ROBO A CUENTAHABIENTE", "dudosa",
     "Robo a persona en la calle, pero en contexto laboral o bancario"),
    ("09", r"TENTATIVA DE EXTORSION", "dudosa",
     "Tentativa: verificar si la ENVIPE cuenta extorsiones no consumadas"),
    ("09", r"^EXTORSION", "alta", ""),
    ("10", r"^AMENAZAS", "alta", ""),
    ("11", r"^LESIONES", "alta", "Solo intencionales (las culposas se excluyeron arriba)"),
    ("14", r"^VIOLACION(?! (DE|A) )", "alta",
     "Excluye 'violación de correspondencia/sellos/…', que no son delitos sexuales"),
    ("13", r"ABUSO SEXUAL|ACOSO SEXUAL|HOSTIGAMIENTO SEXUAL", "alta", ""),
    ("12", r"SECUESTRO|PLAGIO", "alta", ""),
    ("12", r"PRIVACION DE LA LIBERTAD", "dudosa", "Verificar si la ENVIPE lo cuenta como secuestro"),
    ("07", r"USURPACION DE IDENTIDAD|FALSIFICACION DE TITULOS|DOCUMENTOS DE CREDITO|TARJETA", "dudosa",
     "Clonación de tarjetas o uso de identidad: más cerca de fraude bancario"),
    ("07/08", r"^FRAUDE", "dudosa",
     "La FGJ no distingue fraude bancario de fraude al consumidor"),
    ("08", r"^ABUSO DE CONFIANZA", "dudosa", "Verificar contra la definición de fraude al consumidor"),
    ("06", r"^ROBO DE (OBJETOS|DINERO|DOCUMENTOS|CELULAR)|^ROBO$|^ROBO SIMPLE", "dudosa",
     "'Otro tipo de robo' es residual en la ENVIPE"),
    ("—",  r"VIOLENCIA FAMILIAR", "dudosa",
     "La ENVIPE no tiene este tipo; parte podría corresponder a lesiones o amenazas en el hogar"),
]


def proponer(delito):
    """Devuelve (BPCOD, confianza, razón) de la PRIMERA regla que coincide."""
    texto = sin_acentos(delito)
    for codigo, patron, confianza, razon in REGLAS:
        if re.search(patron, texto):
            return codigo, confianza, razon
    return "?", "sin asignar", ""


# --- Catálogo de los 357 delitos: carpetas 2020–2024 y la categoría FGJ más frecuente ---
catalogo = pd.DataFrame({
    "categoria_fgj": fgj.groupby("delito")["categoria_delito"].agg(lambda s: s.mode().iat[0])})
catalogo["carpetas_2020_2024"] = (fgj[periodo].groupby("delito").size()
                                  .reindex(catalogo.index, fill_value=0))
propuestas = [proponer(d) for d in catalogo.index]
catalogo["bpcod_propuesto"] = [p[0] for p in propuestas]
catalogo["confianza"]       = [p[1] for p in propuestas]
catalogo["razon"]           = [p[2] for p in propuestas]
catalogo["delito_envipe"]   = catalogo["bpcod_propuesto"].map(BPCOD_NOMBRE)
catalogo = catalogo.sort_values(["bpcod_propuesto", "carpetas_2020_2024"], ascending=[True, False])

# --- Resumen por BPCOD ---
total = catalogo["carpetas_2020_2024"].sum()
resumen = catalogo.groupby(["bpcod_propuesto", "confianza"])["carpetas_2020_2024"].agg(["size", "sum"])
resumen.columns = ["delitos FGJ", "carpetas 2020–2024"]
resumen["% del total"] = (100 * resumen["carpetas 2020–2024"] / total).round(1)
print(f"Total de carpetas 2020–2024 (solo delitos): {total:,}\n")
print(resumen.to_string())

# --- Detalle: lo asignado (con carpetas) y TODO lo no asignado ---
for codigo in catalogo["bpcod_propuesto"].unique():
    bloque = catalogo[(catalogo["bpcod_propuesto"] == codigo) & (catalogo["carpetas_2020_2024"] > 0)]
    if bloque.empty:
        continue
    print(f"\n===== {codigo} {BPCOD_NOMBRE[codigo]} =====")
    for delito, fila in bloque.iterrows():
        marca = "  " if fila["confianza"] == "alta" else "⚠️"
        print(f"  {marca} {fila['carpetas_2020_2024']:>7,}  {delito}")

# --- Borrador para que tomes la decisión final ---
borrador = catalogo.reset_index().rename(columns={"index": "delito"})
borrador["decision_final"] = ""          # ← lo llenas tú
borrador["justificacion"]  = ""          # ← y aquí el porqué, sobre todo en las dudosas
RUTA_BORRADOR = CARPETA_SALIDA / "correspondencia_bpcod_fgj_borrador.csv"
borrador.to_csv(RUTA_BORRADOR, index=False, encoding="utf-8-sig")
print(f"\nBorrador guardado: {RUTA_BORRADOR.name} ({len(borrador)} delitos)")

1. La ENVIPE solo entrevista a personas de 18 años o más; la Fiscalía registra víctimas de cualquier edad. El caso más claro es "acoso sexual agravado en contra de menores". Lo que es más difícil: "violación equiparada" (5,223) supera a "violación" (4,294). La violación equiparada incluye, entre otros supuestos, a víctimas menores de 12 años. Así que en los numeradores de 11, 13 y 14 hay carpetas de menores que, del lado de la ENVIPE, no existen. Los delitos del hogar (01–04) no tienen este problema.
2. La unidad no es la misma. La ENVIPE cuenta delitos por víctima; el archivo de la FGJ cuenta carpetas. Un asalto en un pesero con cinco pasajeros puede ser una sola carpeta y cinco delitos en la ENVIPE.

Los dos problemas tienen la misma salida probable: en el mismo portal, la Fiscalía publica, si mal no recuerdo, un conjunto de víctimas en carpetas de investigación, una fila por víctima y con su edad. 

3. Violencia familiar (173 mil) es la decisión más importante del numerador. La ENVIPE no tiene ese tipo, pero sí capta lesiones y amenazas cometidas por familiares, dentro de 10 y 11. La Fiscalía, en cambio, registra muchas de esas agresiones como violencia familiar. Si la excluyes, el numerador de 10 y 11 queda subestimado frente a la ENVIPE, y R sale artificialmente baja justo en esos delitos. Lo honesto es dejarla fuera en la versión principal y reportar una cota con ella incluida.

In [ ]:
# Correspondencia v2: ajustes, cotas y tabla para el escrito


# Reglas que deben ganarle a las de la celda 14 (van ANTES)
REGLAS_ANTES = [
    ("—",  r"MENOR|INFANT|^ESTUPRO", "alta",
     "Víctima menor de edad: la ENVIPE solo entrevista a personas de 18 años o más"),
    ("—",  r"^ROBO DE OBJETOS A ESCUELA", "alta", "Víctima institucional"),
    ("—",  r"SERVICIO OFICIAL", "alta", "Vehículo oficial: no pertenece a un hogar"),
    ("06", r"MOTOPATIN", "dudosa", "Monopatín eléctrico: verificar si la ENVIPE lo cuenta como vehículo"),
]
# Reglas para lo que quedó sin asignar (van DESPUÉS: solo atrapan lo que antes no coincidía)
REGLAS_DESPUES = [
    ("06", r"^ROBO EN EVENTOS MASIVOS", "dudosa", "Robo a personas en eventos: entre 05 y 06"),
    ("06", r"^ROBO DE (ALHAJAS|ANIMALES)", "dudosa", "Bienes del hogar o personales"),
    ("—",  r"^ROBO S/V DENTRO DE NEGOCIOS", "dudosa", "Probablemente la víctima es el negocio; podría ser un cliente"),
    ("01", r"^TENTATIVA DE ROBO DE VEHICULO", "dudosa", "Tentativa: verificar si la ENVIPE la cuenta"),
    ("06", r"^TENTATIVA DE ROBO$", "dudosa", "Tentativa sin tipo de robo: asignación residual"),
    ("07/08", r"^TENTATIVA DE FRAUDE", "dudosa", "Tentativa: verificar si la ENVIPE la cuenta"),
    ("14", r"^TENTATIVA DE VIOLACION", "dudosa", "Tentativa: la ENVIPE podría registrarla como abuso sexual (13)"),
    ("12", r"^PRIV\. ILEGAL DE LA LIB", "dudosa", "Privación de la libertad con robo de vehículo: cercano a secuestro exprés"),
    ("13", r"CONTRA LA INTIMIDAD SEXUAL", "dudosa", "Difusión de contenido íntimo: verificar si la ENVIPE lo capta en 13"),
    ("09", r"^COBRANZA ILEGITIMA", "dudosa", "Cobranza con amenazas: cercana a extorsión"),
    ("15", r"^OTROS DELITOS$", "dudosa", "Residual"),
    ("—",  r"^DESPOJO|^ALLANAMIENTO|^ABUSO DE AUTORIDAD", "dudosa",
     "Victimización real sin tipo en la ENVIPE; podría caer en 'otros delitos' (15)"),
]
SIN_REGLA = ("—", "alta", "Fuera del catálogo de la ENVIPE (contra el Estado, la administración pública, el ambiente, etc.)")
TODAS_LAS_REGLAS = REGLAS_ANTES + REGLAS + REGLAS_DESPUES


def proponer_v2(delito):
    texto = sin_acentos(delito)
    for codigo, patron, confianza, razon in TODAS_LAS_REGLAS:
        if re.search(patron, texto):
            return codigo, confianza, razon
    return SIN_REGLA


anterior = catalogo["bpcod_propuesto"].copy()
propuestas = [proponer_v2(d) for d in catalogo.index]
catalogo["bpcod_propuesto"] = [p[0] for p in propuestas]
catalogo["confianza"]       = [p[1] for p in propuestas]
catalogo["razon"]           = [p[2] for p in propuestas]
catalogo["delito_envipe"]   = catalogo["bpcod_propuesto"].map(BPCOD_NOMBRE)

# --- Qué cambió respecto a la v1 (fuera de lo que pasó de "?" a "—" sin regla) ---
cambio = (anterior != catalogo["bpcod_propuesto"]) & (catalogo["razon"] != SIN_REGLA[2])
print("Cambios respecto al borrador v1:")
for delito, fila in catalogo[cambio].sort_values("carpetas_2020_2024", ascending=False).iterrows():
    print(f"  {anterior[delito]:>5} → {fila['bpcod_propuesto']:<5} {fila['carpetas_2020_2024']:>7,}  {delito}")
print(f"\nYa no queda nada sin asignar: {(catalogo['bpcod_propuesto'] == '?').sum()} delitos en '?'")

# --- Tabla para el escrito: cota inferior (alta) y superior (alta + dudosa) por delito ENVIPE ---
en_envipe = ~catalogo["bpcod_propuesto"].isin(["—", "03"])
p5 = (catalogo[en_envipe]
      .assign(alta=lambda t: t["carpetas_2020_2024"].where(t["confianza"] == "alta", 0))
      .groupby("bpcod_propuesto")
      .agg(delitos_fgj=("carpetas_2020_2024", "size"),
           carpetas_alta=("alta", "sum"),
           carpetas_alta_y_dudosa=("carpetas_2020_2024", "sum")))
p5.insert(0, "delito ENVIPE", p5.index.map(BPCOD_NOMBRE))
p5["% dudoso"] = (100 * (1 - p5["carpetas_alta"] / p5["carpetas_alta_y_dudosa"])).round(1)
p5.loc["Total"] = ["", p5["delitos_fgj"].sum(), p5["carpetas_alta"].sum(),
                   p5["carpetas_alta_y_dudosa"].sum(), np.nan]
p5.loc["Total", "% dudoso"] = round(100 * (1 - p5.loc["Total", "carpetas_alta"]
                                           / p5.loc["Total", "carpetas_alta_y_dudosa"]), 1)
print("\n===== Pregunta 5: carpetas 2020–2024 que corresponden a delitos de la ENVIPE =====")
print(p5.to_string())

# --- Lo que queda fuera, para declararlo con números ---
fuera = catalogo[~en_envipe].groupby(["bpcod_propuesto", "confianza"])["carpetas_2020_2024"].sum()
print(f"\nFuera del numerador: {fuera.sum():,} carpetas ({100 * fuera.sum() / total:.1f}% del total)")
print(f"  de ellas, violencia familiar: {catalogo.loc['VIOLENCIA FAMILIAR', 'carpetas_2020_2024']:,}")
print(f"  y vandalismo (03): {catalogo.loc[catalogo['bpcod_propuesto'] == '03', 'carpetas_2020_2024'].sum():,}")

# --- Guardar ---
borrador = catalogo.reset_index().rename(columns={"index": "delito"})
borrador["decision_final"] = ""
borrador["justificacion"]  = ""
RUTA_BORRADOR = CARPETA_SALIDA / "correspondencia_bpcod_fgj_borrador_v2.csv"
borrador.to_csv(RUTA_BORRADOR, index=False, encoding="utf-8-sig")
TABLAS["P5_correspondencia_resumen"] = p5
print(f"\nGuardado: {RUTA_BORRADOR.name}")

- La cota superior es el doble de la inferior (637 mil contra 319 mil carpetas). Con un rango así, un mapa corregido con "todos los delitos" no es defendible.
- Hay un núcleo sólido: 04 robo a casa, 10 amenazas, 11 lesiones y 14 violación tienen 0–1.5% dudoso, y 05 robo en calle o transporte tiene 17%.
- Hay un bloque que depende por completo de decisiones de clasificación: 06 otro robo, 07 y 08 fraudes, 09 extorsión y 12 secuestro tienen 89–100% dudoso.
- 01 y 02 quedan en medio (37% y 46%), por las motocicletas y los objetos del interior del vehículo.

Esto sugiere que el mapa corregido principal se construye solo con el núcleo de alta confianza, y el total con cotas va como análisis complementario. Ojo con la consecuencia: entonces R también tiene que estimarse para ese mismo subconjunto de delitos en la ENVIPE, porque numerador y denominador deben referirse a los mismos delitos. Eso reduce la muestra todavía más, y es otro argumento para el Rao-Yu.

Por otro lado, los órdenes de magnitud no cuadran. En 2020 ocurrieron en la CDMX 3.92 millones de delitos (sin vandalismo), con R = 11.20% × 66.07% = 7.4%. Eso implica que la ENVIPE "ve" unas 290 mil carpetas al año. La Fiscalía registró ese año 202 mil carpetas de todos los delitos, y entre 60 y 125 mil de los delitos que corresponden a la ENVIPE. Es decir, la ENVIPE implica entre 2.3 y 4.5 veces más carpetas de las que la Fiscalía registra. Hay varias explicaciones posibles, y probablemente es una mezcla:

- Las víctimas dicen que hubo carpeta cuando no la hubo. Por ejemplo, confunden una constancia de hechos o un acta especial con una carpeta.
- Una carpeta puede agrupar varios delitos o víctimas, y la FGJ registra un solo delito por carpeta.
- Algunas carpetas se abren en la Fiscalía General de la República, no en la de la CDMX.
- Efecto telescopio en la encuesta: se reportan delitos de fuera del año de referencia.

Esto importa muchísimo. Si R de la ENVIPE está inflada respecto al registro, el mapa corregido (carpetas / R) subestima el delito. Además, k dejaría de ser solo "incumplimiento institucional": también mediría la distancia entre lo que la víctima cree que pasó y lo que la Fiscalía registró. 

In [ ]:
# ¿Cuántas carpetas "ve" la ENVIPE y cuántas registra la FGJ?

# ENVIPE: Σ FAC_DEL de los incidentes con carpeta, CDMX por ocurrencia, sin vandalismo.
# Promedio anual de los años de referencia 2020–2024 (los mismos que en la FGJ).
juntar_fraude = {"07": "07/08", "08": "07/08"}     # la FGJ no los distingue: los juntamos de los dos lados

env = inc[universo & inc["cdmx_ocu"] & inc["carpeta"]]
env_por_delito = (env.groupby([env["BPCOD"].replace(juntar_fraude), "anio_referencia"])["FAC_DEL"]
                     .sum().unstack(fill_value=0))
env_anual = env_por_delito.mean(axis=1)

# FGJ: las cotas de la celda 15, juntando 07, 08 y 07/08, divididas entre 5 años
fgj_cotas = p5.drop("Total")[["carpetas_alta", "carpetas_alta_y_dudosa"]].astype(float)
fgj_cotas.index = fgj_cotas.index.map(lambda c: "07/08" if c in ["07", "08", "07/08"] else c)
fgj_anual = fgj_cotas.groupby(level=0).sum() / 5

comp = pd.DataFrame({
    "ENVIPE: carpetas/año":  env_anual,
    "FGJ: cota inferior/año": fgj_anual["carpetas_alta"],
    "FGJ: cota superior/año": fgj_anual["carpetas_alta_y_dudosa"],
}).fillna(0)
comp.loc["Total"] = comp.sum()
comp["ENVIPE / FGJ sup."] = comp["ENVIPE: carpetas/año"] / comp["FGJ: cota superior/año"]
comp["ENVIPE / FGJ inf."] = comp["ENVIPE: carpetas/año"] / comp["FGJ: cota inferior/año"].replace(0, np.nan)
comp.insert(0, "delito", [BPCOD_NOMBRE.get(c, "") for c in comp.index])

print(comp.round({"ENVIPE: carpetas/año": 0, "FGJ: cota inferior/año": 0, "FGJ: cota superior/año": 0,
                  "ENVIPE / FGJ sup.": 2, "ENVIPE / FGJ inf.": 2}).to_string())

# Contexto: todas las carpetas de la FGJ (cualquier delito) por año
todas_fgj = t_fgj.loc["Total"].mean()
print(f"\nPara comparar: la FGJ registra en promedio {todas_fgj:,.0f} carpetas/año de CUALQUIER delito.")
print(f"La ENVIPE implica {comp.loc['Total', 'ENVIPE: carpetas/año']:,.0f} carpetas/año solo de sus 14 delitos.")

# Año por año, el total (¿la brecha es estable o cambia?)
env_total = env_por_delito.sum()
fgj_sup = fgj[periodo & fgj["delito"].isin(catalogo.index[en_envipe])].groupby("anio_h").size()
print("\nTotal por año de referencia (ENVIPE vs. FGJ cota superior):")
for a in env_total.index:
    print(f"  {a}: ENVIPE {env_total[a]:>9,.0f} · FGJ {fgj_sup.get(a, 0):>8,} · razón {env_total[a] / fgj_sup.get(a, np.nan):.2f}")

TABLAS["P5_envipe_vs_fgj"] = comp.round(2)

La brecha no es un problema de clasificación. En el núcleo sólido, donde la correspondencia no tiene dudas, la ENVIPE implica muchas más carpetas que las que registra la Fiscalía. En cambio, los delitos sexuales y los fraudes casi cuadran: violación 0.87, abuso sexual 1.3 a 1.7, fraudes 0.96 con la cota superior. Para violación la cuadratura es todavía más notable, porque la FGJ incluye a víctimas menores de edad que la ENVIPE no entrevista.

La brecha total es estable, de 1.9 a 2.5 según el año. Eso quiere decir que es sistemática, no ruido. El 2.39 de 2024 está algo inflado por el rezago del corte.

Las explicaciones posibles por delito, que son hipótesis para comprobar y no conclusiones:

- Lesiones y amenazas: la violencia familiar. Si las 34.6 mil carpetas anuales de violencia familiar se sumaran a 10 y 11, la razón conjunta bajaría de alrededor de 3.3 a alrededor de 1.3. Es un indicio fuerte de que la ENVIPE registra como lesiones y amenazas lo que la Fiscalía registra como violencia familiar.
- Robo de vehículo: la jurisdicción. Es el delito con mayor incentivo a denunciar, por el seguro, y aun así es de los que tienen mayor brecha. Una víctima de Ecatepec a la que le roban el coche en la CDMX puede denunciar ante la Fiscalía del Estado de México, y la ENVIPE lo cuenta como carpeta en la CDMX, porque la pregunta dice "MP o Fiscalía Estatal", sin precisar cuál. Esto se prueba con el punto 3C, que viene ahora.
- Robo a casa y en la calle: la confusión entre carpeta y otros trámites. Ahí no se ve un mecanismo tan claro, y la hipótesis de que la víctima confunda una constancia o un acta con una carpeta gana peso.
- "Otros delitos" (15) no tiene contraparte en la FGJ. Habría que sacarlo del mapa corregido de los dos lados.

La implicación para el proyecto, en palabras simples: el k de la ENVIPE es un k declarado por la víctima. El mapa corregido (carpetas registradas / R) supone que R describe las carpetas registradas. Si en lesiones la víctima declara cinco carpetas por cada una que existe, dividir entre R subestima el delito unas cinco veces. Veo tres caminos:

1. Construir el mapa solo con los delitos donde ENVIPE y FGJ cuadran razonablemente.
2. Calibrar R con la razón ENVIPE/FGJ por delito.
3. Tratar esa razón como una tercera barrera: entre lo que la víctima cree que se registró y lo que se registró. Si varía entre alcaldías, puede ser una aportación del trabajo en sí misma.

Pendientes: carpetas de la ENVIPE solo de residentes de la CDMX, violencia familiar como prueba de sensibilidad, el conjunto de víctimas de la FGJ con edad, y el error de muestreo de los totales de la ENVIPE por delito.

In [ ]:
#  Parte 3B (alcaldía 999) y 3C (víctimas no residentes)

cdmx_u = universo & inc["cdmx_ocu"]          # CDMX por ocurrencia, sin vandalismo (incluye los 999)
w = inc["FAC_DEL"]

filas = []
for a in ANIOS:
    s     = cdmx_u & (inc["anio_envipe"] == a)
    s999  = s & inc["excl_alc_999"]
    nores = s & inc["no_residente"]
    carp  = s & inc["carpeta"]
    filas.append({
        "edición": ETIQUETA_ANIO[a],
        # --- 3B: sin alcaldía ---
        "incidentes CDMX":       int(s.sum()),
        "incidentes 999":        int(s999.sum()),
        "% incidentes 999":      100 * s999.sum() / s.sum(),
        "delitos CDMX":          w[s].sum(),
        "delitos 999":           w[s999].sum(),
        "% delitos 999":         100 * w[s999].sum() / w[s].sum(),
        # --- 3C: no residentes ---
        "incidentes no resid.":  int(nores.sum()),
        "% incidentes no resid.": 100 * nores.sum() / s.sum(),
        "% delitos no resid.":   100 * w[nores].sum() / w[s].sum(),
        "% carpetas no resid.":  100 * w[nores & carp].sum() / w[carp].sum(),
    })
p3 = pd.DataFrame(filas).set_index("edición")

print("===== 3B: incidentes con alcaldía no especificada (BP1_3C = 999) =====")
print(p3[["incidentes CDMX", "incidentes 999", "% incidentes 999",
          "delitos CDMX", "delitos 999", "% delitos 999"]]
      .round(1).to_string(float_format=lambda x: f"{x:,.1f}"))
print("Control contra el diccionario: 2021 → 78,625 delitos 999 (2.0%) · 2022 → 63,779")

print("\n===== 3C: víctimas que no residen en la CDMX =====")
print(p3[["incidentes no resid.", "% incidentes no resid.", "% delitos no resid.",
          "% carpetas no resid."]].round(1).to_string())
print("Control contra el diccionario: 2022 → 210 de 3,029 incidentes")

# ¿De dónde vienen? (cinco años juntos, ponderado)
de_fuera = inc[cdmx_u & inc["no_residente"]]
origen = (100 * de_fuera.groupby("ENT_RES")["FAC_DEL"].sum() / de_fuera["FAC_DEL"].sum()).sort_values(ascending=False)
print("\nEntidad de residencia de las víctimas no residentes (% de sus delitos, 2020–2024):")
print(origen.head(5).round(1).to_string())

# La prueba de la hipótesis de jurisdicción: ¿los no residentes pesan más en las carpetas que en los delitos?
# Por tipo de delito, cinco años juntos
por_delito = pd.DataFrame({
    "% delitos no resid.":  100 * inc[cdmx_u & inc["no_residente"]].groupby("BPCOD")["FAC_DEL"].sum()
                            / inc[cdmx_u].groupby("BPCOD")["FAC_DEL"].sum(),
    "% carpetas no resid.": 100 * inc[cdmx_u & inc["no_residente"] & inc["carpeta"]].groupby("BPCOD")["FAC_DEL"].sum()
                            / inc[cdmx_u & inc["carpeta"]].groupby("BPCOD")["FAC_DEL"].sum(),
}).fillna(0)
por_delito.insert(0, "delito", por_delito.index.map(BPCOD_NOMBRE))
print("\nNo residentes por tipo de delito (2020–2024, ponderado):")
print(por_delito.round(1).to_string())

TABLAS["P3B_sin_alcaldia"]   = p3.iloc[:, :6].round(1)
TABLAS["P3C_no_residentes"]  = p3.iloc[:, 6:].round(1)
TABLAS["P3C_no_resid_delito"] = por_delito.round(1)

### Casos sin alcaldía

Pesan alrededor de 2% de los delitos de la CDMX, de forma estable, de 1.9% a 2.0% entre 2020 y 2023. En la edición 2025 suben a 2.5%. En incidentes y en delitos el peso es casi el mismo, así que los 999 no tienen factores atípicos. Es una exclusión acotada. Las carpetas sin alcaldía pasan de 0.3–0.6% a 3.1–3.7% en 2023–2024. En los últimos años, numerador y denominador pierden una fracción parecida por falta de alcaldía.

### Víctimas no residentes

Son entre 7% y 9.5% de los incidentes, pero entre 10% y 15% de los delitos expandidos, y la cifra va subiendo. La diferencia indica que los no residentes llegan con factores más altos que el promedio: se muestrearon en sus propios estados, sobre todo el Estado de México (75.5%), con factores de expansión mayores. Este grupo es el que hace diferir las dos geografías. La estimación por residencia lo excluye, y en su lugar incluye a residentes de la CDMX victimizados fuera.

La hipótesis de jurisdicción no se sostiene. Si los no residentes estuvieran denunciando en su estado, pesarían más en las carpetas que en los delitos. Pasa lo contrario: en total pesan menos en carpetas (7–13%) que en delitos (10–15%). En robo de vehículo los dos pesos son casi iguales (13.6% y 14.1%). Aun suponiendo que todas las carpetas de no residentes se hubieran abierto en otro estado, la razón ENVIPE/FGJ en robo de vehículo solo bajaría de 4.6 a alrededor de 4.0. La jurisdicción explica, cuando mucho, una parte chica de la brecha. Queda como explicación más fuerte la diferencia entre la carpeta que la víctima declara y la que la Fiscalía registra, junto con la violencia familiar en lesiones y amenazas.

Una advertencia sobre el desglose por delito: en violación (31.9% de los delitos y 0% de las carpetas) y secuestro hay muy pocos casos de no residentes, así que esas cifras no deben leerse.

In [ ]:
# UPM compartidas, covarianza entre alcaldías y ρ entre geografías

def matriz_z(inc_a, upms, num, den, geografia):
    """Las z de Taylor de las 16 alcaldías a la vez: una fila por UPM, una columna por alcaldía.
    Cada columna es exactamente lo que razon_rapida calcula por dentro."""
    w = inc_a["FAC_DEL"]
    columnas = {}
    for clave in NOMBRE_ALC.index:
        dominio = ~inc_a["excl_vandalismo"] & (inc_a[geografia] == clave)
        y = w * inc_a[num] * dominio
        x = w * inc_a[den] * dominio
        if x.sum() == 0:                       # alcaldía sin casos: columna de ceros
            columnas[clave] = pd.Series(0.0, index=upms)
            continue
        razon = y.sum() / x.sum()
        z = (y - razon * x) / x.sum()
        columnas[clave] = z.groupby([inc_a["EST_DIS"], inc_a["UPM_DIS"]]).sum().reindex(upms, fill_value=0.0)
    return pd.DataFrame(columnas)


def covarianza_taylor(Z):
    """Matriz de covarianzas 16 × 16. Misma fórmula que la varianza, pero cruzando columnas:
    centrar dentro de cada estrato, escalar por √(n_h/(n_h−1)) y multiplicar Zᵀ·Z."""
    estrato = pd.Series(Z.index.get_level_values("EST_DIS"))
    n_h = estrato.map(estrato.value_counts()).to_numpy()
    centrada = Z - Z.groupby(level="EST_DIS").transform("mean")
    centrada = centrada.mul(np.sqrt(n_h / (n_h - 1)), axis=0)
    return centrada.T @ centrada


filas_3a, filas_rho = [], []
for a in ANIOS:
    inc_a = inc[inc["anio_envipe"] == a].assign(todos=True)

    # UPM compartidas: aportan incidentes (sin vandalismo) a 2 o más alcaldías de ocurrencia
    base_a = inc_a[~inc_a["excl_vandalismo"] & inc_a["alc_ocu"].notna()]
    alc_por_upm = base_a.groupby(["EST_DIS", "UPM_DIS"])["alc_ocu"].nunique()
    compartidas = alc_por_upm.index[alc_por_upm >= 2]

    for param, (num, den) in PARAMETROS.items():
        Z = matriz_z(inc_a, UPMS[a], num, den, "alc_ocu")
        S = covarianza_taylor(Z)
        var = np.diag(S)

        # Control: la diagonal debe ser el EE² de la celda 5
        ee_c5 = (diag[(diag["anio"] == a) & (diag["param"] == param)]
                 .set_index("alc")["ee"].reindex(NOMBRE_ALC.index).to_numpy())
        ok = np.isfinite(ee_c5) & (ee_c5 > 0)
        dif = np.max(np.abs(np.sqrt(var[ok]) - ee_c5[ok]) / ee_c5[ok])

        # Correlaciones entre alcaldías (fuera de la diagonal)
        sd = np.sqrt(var)
        with np.errstate(invalid="ignore", divide="ignore"):
            corr = S.to_numpy() / np.outer(sd, sd)
        fuera_diag = corr[~np.eye(16, dtype=bool)]
        fuera_diag = fuera_diag[np.isfinite(fuera_diag)]

        # Varianza que viene solo de las UPM compartidas (apagando las exclusivas)
        # 1.0 en las filas de UPM compartidas, 0.0 en las exclusivas; mul(..., axis=0) multiplica fila por fila
        Z_comp = Z.mul(Z.index.isin(compartidas).astype(float), axis=0)
        var_comp = np.diag(covarianza_taylor(Z_comp))
        with np.errstate(invalid="ignore", divide="ignore"):
            fraccion = var_comp / var

        filas_3a.append({
            "edición": ETIQUETA_ANIO[a], "parámetro": param,
            "UPM compartidas": len(compartidas), "UPM con incidentes": len(alc_por_upm),
            "|corr| mediana": np.median(np.abs(fuera_diag)),
            "|corr| máxima":  np.max(np.abs(fuera_diag)),
            "% var. de UPM compartidas (mediana)": 100 * np.nanmedian(fraccion),
            "% var. de UPM compartidas (máx.)":    100 * np.nanmax(fraccion),
            "control EE (dif. máx.)": dif,
        })

        # --- Robustez: misma estimación por RESIDENCIA y ρ de Spearman con ocurrencia ---
        ocu = diag[(diag["anio"] == a) & (diag["param"] == param)].set_index("alc")["est"]
        res = pd.Series({c: razon_rapida(inc_a, UPMS[a], num, den,
                                         ~inc_a["excl_vandalismo"] & (inc_a["alc_res"] == c))[0]
                         for c in NOMBRE_ALC.index})
        par = pd.DataFrame({"ocurrencia": ocu, "residencia": res}).dropna()
        filas_rho.append({"edición": ETIQUETA_ANIO[a], "parámetro": param,
                          "ρ Spearman": par.corr(method="spearman").iloc[0, 1],
                          "alcaldías comparadas": len(par)})
        for c in NOMBRE_ALC.index:
            filas_rho.append({"edición": ETIQUETA_ANIO[a], "parámetro": param, "alc": c,
                              "ocurrencia": ocu.get(c), "residencia": res.get(c)})

t3a = pd.DataFrame(filas_3a).set_index(["edición", "parámetro"])
print("===== 3A: UPM compartidas y correlación de errores entre alcaldías (ocurrencia) =====")
print(t3a.round(3).to_string())

rho = pd.DataFrame([f for f in filas_rho if "ρ Spearman" in f]).set_index(["edición", "parámetro"])
print("\n===== Robustez: ρ de Spearman entre ocurrencia y residencia (estimaciones DIRECTAS) =====")
print(rho.round(2).to_string())

# Las 80 celdas alcaldía-año juntas, por parámetro
celdas = pd.DataFrame([f for f in filas_rho if "alc" in f])
for param in ["d", "k"]:
    sub = celdas[celdas["parámetro"] == param].dropna(subset=["ocurrencia", "residencia"])
    print(f"ρ con las {len(sub)} celdas alcaldía-año de {param}: "
          f"{sub[['ocurrencia', 'residencia']].corr(method='spearman').iloc[0, 1]:.2f}")

TABLAS["P3A_upm_compartidas"] = t3a.round(3)
TABLAS["P3A_spearman_geografias"] = rho.round(2)
estimaciones_residencia = celdas      # las guardamos: el modelo las va a necesitar

El control salió en 0.0: la diagonal de la matriz reproduce exactamente los errores estándar de la celda 5, así que las covarianzas son confiables. Y las 278 UPM compartidas de 2022 coinciden con el diccionario.

Entre 25% y 31% de las UPM aportan incidentes a más de una alcaldía, y en la alcaldía típica esas UPM generan entre 30% y 53% de su varianza, pero la correlación entre los errores de alcaldías distintas es prácticamente cero: la mediana del valor absoluto va de 0.000 a 0.005, y el par más correlacionado de todo el panel llega a 0.22. ¿Cómo puede una UPM aportar la mitad de la varianza y no correlacionar nada? Porque, dentro de una UPM compartida, los incidentes que van a Iztapalapa y los que van a Coyoacán son, en general, de personas distintas, con denuncias distintas. Sus z no tienden a moverse en la misma dirección. Para que haya covarianza, la UPM tendría que empujar a las dos alcaldías hacia arriba (o hacia abajo) a la vez, y eso casi no pasa.

La medida relevante para el Fay-Herriot es la correlación, no la fracción de varianza. Entre 25% y 31% de las UPM aportan incidentes a más de una alcaldía de ocurrencia. Sin embargo, la correlación entre los errores de muestreo de alcaldías distintas, estimada con la matriz de covarianzas de Taylor, es despreciable (mediana de |ρ| ≤ 0.005; máximo 0.22), por lo que el supuesto de independencia del modelo se sostiene de manera aproximada. Así el costo de elegir la ocurrencia queda acotado con números.

La fracción de varianza de UPM compartidas puede ir en una nota al pie, aclarando que es aproximada porque omite los términos cruzados.

La ρ entre geografías

Con las 80 celdas juntas: 0.75 para d y 0.78 para k. Año por año varía mucho, de 0.34 a 0.90.
Es menor que la de Buil-Gil (0.91 y 0.88), pero no es comparable todavía. Ellos comparan estimaciones del modelo, y nosotros, estimaciones directas con CV de 30% o más en muchas celdas. Ese ruido baja la correlación aunque las dos geografías midan lo mismo. El 0.34 de k en 2025 es el ejemplo: es el año con menos denuncias y con dos celdas de cinco casos o menos. Las dos geografías comparten la mayoría de los incidentes, los de residentes victimizados en su propia alcaldía, así que parte de la correlación es mecánica. Buil-Gil tiene el mismo matiz.

Con estimaciones directas, ρ = 0.75 (d) y 0.78 (k); se recalcula con las estimaciones del modelo en la sección X.

In [ ]:
# Exportar las tablas del escrito y las estimaciones directas

import importlib
importlib.invalidate_caches()        # vuelve a revisar qué paquetes hay instalados (por si acabas de instalar uno)
try:
    import openpyxl                  # pandas lo usa para escribir .xlsx
except ImportError:
    raise ImportError("Falta openpyxl: corre `%pip install openpyxl` en otra celda y vuelve a correr esta")

# --- 1. Las tablas del escrito, una hoja por tabla ---
DESCRIPCION = {
    "P1_incidentes_por_alcaldia":  "Incidentes muestrales por alcaldía de ocurrencia y edición, sin vandalismo",
    "P2_upm_por_alcaldia":         "UPM distintas que aportan incidentes a cada alcaldía",
    "P3_deff_d":                   "Efecto de diseño de d (varianza de Taylor / varianza de MAS)",
    "P3_deff_k":                   "Efecto de diseño de k",
    "P3_cv_directo_d":             "CV (%) de la estimación directa de d",
    "P3_cv_directo_k":             "CV (%) de la estimación directa de k",
    "P4_denuncias_ocurrencia":     "Incidentes denunciados (denominador de k), por ocurrencia",
    "P4_denuncias_residencia":     "Incidentes denunciados, por residencia",
    "P4_upm_con_denuncia":         "UPM con al menos una denuncia",
    "P5_correspondencia_resumen":  "Carpetas FGJ 2020–2024 por delito ENVIPE: cota inferior (alta) y superior (alta+dudosa)",
    "P5_envipe_vs_fgj":            "Carpetas por año: implicadas por la ENVIPE vs. registradas por la FGJ",
    "P6_razones_cdmx":             "Razones de no denuncia (BP1_23), CDMX, % de los no denunciados",
    "P6_razones_region":           "Razones de no denuncia por región, promedio 2020–2024",
    "P6_cv_region":                "CV (%) de las razones por región",
    "P7_bp1_24_por_anio":          "BP1_24 sin dato entre denunciados, por año",
    "P7_bp1_24_por_alcaldia":      "BP1_24 sin dato por alcaldía y sensibilidad de k",
    "P3A_upm_compartidas":         "UPM compartidas y correlación de errores entre alcaldías",
    "P3A_spearman_geografias":     "ρ de Spearman ocurrencia vs. residencia (estimaciones directas)",
    "P3B_sin_alcaldia":            "Peso de los incidentes con alcaldía 999",
    "P3C_no_residentes":           "Peso de las víctimas no residentes",
    "P3C_no_resid_delito":         "Víctimas no residentes por tipo de delito",
}
faltan_desc = set(TABLAS) - set(DESCRIPCION)
if faltan_desc:
    print("Tablas sin descripción:", faltan_desc)

RUTA_EXCEL = CARPETA_SALIDA / "dia3_tablas_diagnostico.xlsx"
with pd.ExcelWriter(RUTA_EXCEL) as excel:                   # "with" cierra el archivo al terminar
    indice = pd.DataFrame({"hoja": list(TABLAS),
                           "contenido": [DESCRIPCION.get(n, "") for n in TABLAS]})
    indice.to_excel(excel, sheet_name="Índice", index=False)
    for nombre, tabla in TABLAS.items():
        tabla.to_excel(excel, sheet_name=nombre[:31])       # Excel no acepta nombres de hoja de más de 31 caracteres
    # La correspondencia completa (357 delitos) también va aquí, como respaldo de la tabla P5
    catalogo.reset_index().rename(columns={"index": "delito"}).to_excel(
        excel, sheet_name="P5_correspondencia_completa", index=False)
print(f"Guardado: {RUTA_EXCEL.name} ({len(TABLAS) + 2} hojas)")

# --- 2. Estimaciones directas por alcaldía, las dos geografías: insumo del modelo (día 9) ---
directas = pd.concat([
    diag.assign(geografia="ocurrencia")[["geografia", "anio", "alc", "param", "n", "est", "ee", "cv", "deff"]],
    estimaciones_residencia.dropna(subset=["alc"])
        .melt(id_vars=["edición", "parámetro", "alc"], value_vars=["residencia"],
              var_name="geografia", value_name="est")
        .assign(anio=lambda t: t["edición"].str[:4].astype(int))
        .rename(columns={"parámetro": "param"})[["geografia", "anio", "alc", "param", "est"]],
], ignore_index=True)
RUTA_DIRECTAS = CARPETA_SALIDA / "estimaciones_directas.parquet"
directas.to_parquet(RUTA_DIRECTAS, index=False)
print(f"Guardado: {RUTA_DIRECTAS.name} ({len(directas)} filas)")
print("Nota: por residencia guardamos solo la estimación; su EE se calcula en el día 9 con razon_rapida.")